# Neural Degeneration Early-Warning System (NDEWS)

**Self-contained Colab notebook.** Predicts training instability / representational
collapse in CNNs by extracting 6 internal signals via PyTorch hooks, then training a
Random-Forest forecaster on sliding windows of those signals.

This notebook bundles the entire `src/` + `analysis/` codebase into one file.

**How to run:** `Runtime -> Run all`. For a GPU: `Runtime -> Change runtime type -> GPU`.

Pipeline: install deps -> define library -> quick smoke test -> batch experiments ->
leave-one-run-out evaluation vs. baselines -> signal lead-time analysis -> plots.


## 0. Install dependencies

In [ ]:
# Colab already ships torch/torchvision/sklearn/matplotlib; install is a no-op there.
# Uncomment if running on a bare environment.
# !pip install -q torch torchvision scikit-learn matplotlib tqdm numpy
import torch, torchvision, sklearn, numpy, matplotlib
print('torch', torch.__version__, '| torchvision', torchvision.__version__,
      '| sklearn', sklearn.__version__)


## 1. Library code

Each cell below is one module from the project, inlined verbatim (intra-project
imports and CLI blocks stripped). Run them top to bottom.

### 1.1 `seed_utils` — deterministic seeding

In [ ]:
from __future__ import annotations

"""
src/seed_utils.py
=================
Deterministic seeding for fully reproducible training runs.

Usage
-----
    seed_everything(42)

Notes
-----
- ``cudnn.deterministic = True`` disables non-deterministic cuDNN algorithms.
  This may reduce GPU throughput by ~10–20% on some operations.
- ``cudnn.benchmark = False`` prevents cuDNN from selecting the fastest
  algorithm based on input shape, which would reintroduce non-determinism.
- Call this function before constructing DataLoaders, models, or optimisers
  so that all random state is initialised from the same seed.
"""


import random

import numpy as np
import torch


def seed_everything(seed: int) -> None:
    """
    Seed Python, NumPy, and PyTorch RNGs for deterministic reproduction.

    Parameters
    ----------
    seed : int
        The master seed. Every source of randomness is derived from this value.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### 1.2 `model` — SimpleCNN / DeepCNN / TestMLP

In [ ]:
from __future__ import annotations

"""
src/model.py
============
CNN and MLP architectures used in the Training Instability Prediction project.

Models
------
SimpleCNN  : Baseline 2-layer CNN (CIFAR-10/100, 32x32 input).
TestMLP    : Lightweight fully-connected model for quick hook testing.
DeepCNN    : Deeper 3-layer CNN for stress-testing signal extraction.

All image models accept ``num_classes`` so they can be used with datasets
other than CIFAR-10 (e.g. CIFAR-100 with num_classes=100).
"""


import torch.nn as nn
import torch.nn.functional as F


class SimpleCNN(nn.Module):
    """
    Baseline 2-layer convolutional network for CIFAR-10/100.

    Architecture
    ------------
    conv1 (3->32)  -> ReLU -> MaxPool(2)  ->  [B, 32, 16, 16]
    conv2 (32->64) -> ReLU -> MaxPool(2)  ->  [B, 64,  8,  8]
    fc1   (4096->256) -> ReLU
    fc2   (256->num_classes)   -> logits

    Hook targets (recommended): ``"conv2"``, ``"fc1"``
    """

    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 8 * 8, 256)
        self.fc2   = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


class TestMLP(nn.Module):
    """
    Minimal 2-layer MLP — used for fast unit-testing of hook infrastructure.

    Input is flattened from (B, 3, 32, 32) -> (B, 3072).

    Hook targets (recommended): ``"fc1"``
    """

    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(3 * 32 * 32, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


class DeepCNN(nn.Module):
    """
    Deeper 3-layer CNN — used to verify that hooks scale across model depth.

    Architecture
    ------------
    conv1 (3-> 32)  -> ReLU -> MaxPool(2)  ->  [B,  32, 16, 16]
    conv2 (32-> 64) -> ReLU -> MaxPool(2)  ->  [B,  64,  8,  8]
    conv3 (64->128) -> ReLU -> MaxPool(2)  ->  [B, 128,  4,  4]
    fc1   (2048->256) -> ReLU
    fc2   (256->num_classes)   -> logits

    Hook targets (recommended): ``"conv3"``, ``"fc1"``
    """

    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(3,  32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(128 * 4 * 4, 256)
        self.fc2   = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

### 1.3 `regimes` — experiment configs + model builder

In [ ]:
from __future__ import annotations

"""
src/regimes.py
==============
Experiment regime definitions and model construction utilities.

This module is the single source of truth for:
- What a "regime" is (RegimeConfig dataclass)
- What regimes exist (REGIME_REGISTRY)
- How to build a model by name (build_model)
- How to resolve target hook layers (resolve_target_layers)

Previously this logic lived in experiments/baseline__run.py and was imported
by run_many_regimes.py via private-function imports — a coupling that broke
whenever baseline__run.py was refactored.  All experiment scripts now import
from here.

Regime notes
------------
normal               : Clean CIFAR-10 baseline.
label_noise          : 35 % random label corruption — accuracy ceiling, not
                       genuine instability (use as a contrast case only).
over_regularization  : High weight decay suppresses learning.
high_learning_rate   : Adam lr=0.05 causes instability relatively quickly.
overtraining         : Long run on full dataset — memorisation / overfitting.
class_imbalance      : 5 classes downsampled to 20 % keep-probability.
reduced_dataset_size : Only 20 % of training data retained.
delayed_collapse     : **Realistic instability scenario.**  Trains normally
                       for 10 epochs then applies a 50× LR multiplier.
                       Produces a run that looks healthy before it fails —
                       the primary test case for early-warning capability.
warm_then_overfit    : Long run with no regularisation; the model warms up
                       and then progressively memorises, causing val accuracy
                       to plateau then gently degrade.
"""


from dataclasses import dataclass, field

import torch.nn as nn



# ---------------------------------------------------------------------------
# Regime configuration dataclass
# ---------------------------------------------------------------------------

@dataclass
class RegimeConfig:
    """
    Full specification for one training experiment regime.

    Parameters
    ----------
    epochs : int
        Total number of training epochs.
    lr : float
        Initial learning rate for the Adam optimiser.
    weight_decay : float
        L2 regularisation coefficient.
    label_noise : float
        Fraction of training labels randomly replaced [0, 1].
    class_imbalance : float
        Keep-probability for ``imbalance_classes`` [0, 1].  1.0 = no effect.
    train_fraction : float
        Fraction of training data to retain after imbalance filtering [0, 1].
    lr_boost_at_epoch : int | None
        If set, the learning rate is multiplied by ``lr_boost_factor`` at the
        start of this epoch.  Used by ``delayed_collapse`` to simulate a
        training catastrophe after a healthy warm-up period.
    lr_boost_factor : float
        Multiplier applied when ``lr_boost_at_epoch`` is reached.
    augment : bool
        Whether to apply train-time data augmentation.  Set ``False`` for
        memorization/overfitting regimes (augmentation suppresses overfitting).
    model : str | None
        Architecture this regime pins (``"simple"`` or ``"deep"``).  ``None``
        means the experiment runner chooses (its global ``--model`` default).
    """
    epochs: int
    lr: float
    weight_decay: float
    label_noise: float
    class_imbalance: float
    train_fraction: float
    lr_boost_at_epoch: int | None = field(default=None)
    lr_boost_factor: float = field(default=10.0)
    augment: bool = field(default=True)
    model: str | None = field(default=None)


# ---------------------------------------------------------------------------
# Regime registry
# ---------------------------------------------------------------------------

REGIME_REGISTRY: dict[str, RegimeConfig] = {
    # ---- standard baselines ----
    "normal": RegimeConfig(
        epochs=20,
        lr=1e-3,
        weight_decay=0.0,
        label_noise=0.0,
        class_imbalance=1.0,
        train_fraction=1.0,
    ),
    "label_noise": RegimeConfig(
        epochs=20,
        lr=1e-3,
        weight_decay=0.0,
        label_noise=0.35,
        class_imbalance=1.0,
        train_fraction=1.0,
    ),
    "over_regularization": RegimeConfig(
        epochs=20,
        lr=1e-3,
        weight_decay=0.10,
        label_noise=0.0,
        class_imbalance=1.0,
        train_fraction=1.0,
    ),
    "high_learning_rate": RegimeConfig(
        epochs=20,
        lr=0.05,
        weight_decay=0.0,
        label_noise=0.0,
        class_imbalance=1.0,
        train_fraction=1.0,
    ),
    "overtraining": RegimeConfig(
        epochs=80,
        lr=1e-3,
        weight_decay=0.0,
        label_noise=0.0,
        class_imbalance=1.0,
        train_fraction=1.0,
    ),
    "class_imbalance": RegimeConfig(
        epochs=20,
        lr=1e-3,
        weight_decay=0.0,
        label_noise=0.0,
        class_imbalance=0.20,
        train_fraction=1.0,
    ),
    "reduced_dataset_size": RegimeConfig(
        epochs=20,
        lr=1e-3,
        weight_decay=0.0,
        label_noise=0.0,
        class_imbalance=1.0,
        train_fraction=0.20,
    ),

    # ---- research-grade instability scenarios ----
    #
    # delayed_collapse
    # ----------------
    # Trains at lr=1e-3 for 10 epochs (healthy ascent), then multiplies
    # lr by 50× at epoch 11, causing gradient explosion / loss divergence.
    # This is the PRIMARY test case: the model is on-track when it fails,
    # so any predictor must use internal signals — not just "did training
    # start badly?" — to detect the impending catastrophe.
    "delayed_collapse": RegimeConfig(
        epochs=30,
        lr=1e-3,
        weight_decay=0.0,
        label_noise=0.0,
        class_imbalance=1.0,
        train_fraction=1.0,
        lr_boost_at_epoch=11,
        lr_boost_factor=50.0,
    ),

    # warm_then_overfit
    # -----------------
    # Long run with no regularisation on full data.  The model warms up
    # cleanly, then progressively memorises training examples.  Val accuracy
    # plateaus and slowly degrades — a gentler instability than
    # delayed_collapse but representative of real production failures.
    "warm_then_overfit": RegimeConfig(
        epochs=60,
        lr=1e-3,
        weight_decay=0.0,
        label_noise=0.0,
        class_imbalance=1.0,
        train_fraction=1.0,
    ),

    # ---- endogenous-collapse regimes (genuine forecasting study) ----
    #
    # These produce a GRADUAL representational/overfitting failure: the model
    # warms up, validation accuracy peaks, then sustainedly degrades while the
    # internal signals (effective rank, feature reuse, sparsity, ...) drift in
    # the epochs BEFORE the crash.  Unlike delayed_collapse / high_learning_rate
    # (exogenous shocks with no precursor), the lead-up is forecastable.  All
    # disable augmentation and use a small train set so memorization is fast and
    # reliable on a Colab GPU.
    #
    # memorization_collapse
    # ---------------------
    # SimpleCNN memorizes a tiny clean subset; val peaks early then degrades.
    "memorization_collapse": RegimeConfig(
        epochs=40,
        lr=1e-3,
        weight_decay=0.0,
        label_noise=0.0,
        class_imbalance=1.0,
        train_fraction=0.05,
        augment=False,
    ),

    # label_noise_collapse
    # --------------------
    # Model fits the clean labels first, then memorizes the noisy ones — a
    # classic peak-then-degrade curve with an internal precursor.
    "label_noise_collapse": RegimeConfig(
        epochs=40,
        lr=1e-3,
        weight_decay=0.0,
        label_noise=0.50,
        class_imbalance=1.0,
        train_fraction=0.10,
        augment=False,
    ),

    # deep_memorization
    # -----------------
    # DeepCNN has more capacity → overfits a tiny set harder and earlier.
    "deep_memorization": RegimeConfig(
        epochs=40,
        lr=1e-3,
        weight_decay=0.0,
        label_noise=0.0,
        class_imbalance=1.0,
        train_fraction=0.05,
        augment=False,
        model="deep",
    ),
}

ALL_REGIMES: list[str] = list(REGIME_REGISTRY.keys())


def get_regime_config(name: str) -> RegimeConfig:
    """Return the ``RegimeConfig`` for *name*, or raise ``KeyError``."""
    if name not in REGIME_REGISTRY:
        raise KeyError(
            f"Unknown regime: {name!r}. "
            f"Available regimes: {sorted(REGIME_REGISTRY)}"
        )
    return REGIME_REGISTRY[name]


# ---------------------------------------------------------------------------
# Model construction
# ---------------------------------------------------------------------------

def build_model(name: str, num_classes: int = 10) -> nn.Module:
    """
    Construct a model by name.

    Parameters
    ----------
    name : str
        ``"simple"`` → :class:`SimpleCNN`,  ``"deep"`` → :class:`DeepCNN`.
    num_classes : int
        Number of output logits.  Default 10 (CIFAR-10).
        Pass 100 for CIFAR-100 experiments.
    """
    if name == "deep":
        return DeepCNN(num_classes=num_classes)
    if name == "simple":
        return SimpleCNN(num_classes=num_classes)
    raise ValueError(f"Unknown model name: {name!r}. Choose 'simple' or 'deep'.")


# ---------------------------------------------------------------------------
# Layer resolution
# ---------------------------------------------------------------------------

_DEFAULT_LAYERS: dict[str, list[str]] = {
    "simple": ["conv2", "fc1"],
    "deep":   ["conv3", "fc1"],
}


def resolve_target_layers(
    model_name: str,
    layers_arg: str | None,
) -> list[str]:
    """
    Return the list of layer names to hook.

    Parameters
    ----------
    model_name : str
        ``"simple"`` or ``"deep"``.
    layers_arg : str | None
        Comma-separated layer names from CLI (e.g. ``"conv2,fc1"``).
        ``None`` falls back to the defaults for *model_name*.
    """
    if layers_arg:
        return [s.strip() for s in layers_arg.split(",") if s.strip()]
    return _DEFAULT_LAYERS.get(model_name, ["fc1"])

### 1.4 `dataset` — CIFAR-10 loaders with stress controls

In [ ]:
from __future__ import annotations

"""
src/dataset.py
==============
CIFAR-10 data loading utilities with optional stress-regime controls.

Supported train-set perturbations:
- Label noise
- Class imbalance
- Reduced dataset size
"""


import os
import random
from typing import Iterable

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

# CIFAR-10 per-channel mean/std for [-1, 1] normalisation
_MEAN = (0.5, 0.5, 0.5)
_STD = (0.5, 0.5, 0.5)

# Windows does not support forked workers safely by default.
_DEFAULT_WORKERS = 0 if os.name == "nt" else 2


def _validate_range(name: str, value: float, lo: float, hi: float) -> None:
    if not (lo <= value <= hi):
        raise ValueError(f"{name} must be in [{lo}, {hi}], got {value}")


def _apply_label_noise(
    targets: list[int],
    noise_prob: float,
    n_classes: int,
    seed: int,
) -> list[int]:
    if noise_prob <= 0.0:
        return targets

    rng = random.Random(seed)
    noisy = list(targets)
    for i, old_label in enumerate(noisy):
        if rng.random() < noise_prob:
            # Sample a different class than the current label.
            candidate = rng.randrange(n_classes - 1)
            noisy[i] = candidate if candidate < old_label else candidate + 1
    return noisy


def _build_subset_indices(
    targets: list[int],
    class_imbalance: float,
    imbalance_classes: Iterable[int],
    train_fraction: float,
    seed: int,
) -> list[int]:
    rng = random.Random(seed)
    indices = list(range(len(targets)))

    if class_imbalance < 1.0:
        minority_set = set(imbalance_classes)
        kept: list[int] = []
        for idx in indices:
            label = int(targets[idx])
            if label in minority_set and rng.random() > class_imbalance:
                continue
            kept.append(idx)
        indices = kept

    if train_fraction < 1.0:
        subset_size = max(1, int(len(indices) * train_fraction))
        indices = rng.sample(indices, subset_size)
        indices.sort()

    return indices


def get_cifar_loaders(
    batch_size: int = 128,
    data_root: str = "./data",
    num_workers: int = _DEFAULT_WORKERS,
    pin_memory: bool = True,
    *,
    label_noise: float = 0.0,
    class_imbalance: float = 1.0,
    imbalance_classes: tuple[int, ...] = (0, 1, 2, 3, 4),
    train_fraction: float = 1.0,
    augment: bool = True,
    seed: int = 42,
) -> tuple[DataLoader, DataLoader]:
    """
    Return ``(train_loader, val_loader)`` for CIFAR-10.

    Parameters
    ----------
    batch_size : int
        Mini-batch size for both loaders.
    data_root : str
        Directory where CIFAR-10 is downloaded / cached.
    num_workers : int
        DataLoader worker processes.
    pin_memory : bool
        Use pinned memory for faster GPU transfer when CUDA is available.
    label_noise : float
        Probability of random label replacement on train samples.
    class_imbalance : float
        Keep probability for classes listed in ``imbalance_classes``.
        Set to 1.0 for no class imbalance.
    imbalance_classes : tuple[int, ...]
        Class ids to downsample when ``class_imbalance < 1``.
    train_fraction : float
        Fraction of train data to retain after imbalance filtering.
    augment : bool
        Apply train-time augmentation (RandomCrop + RandomHorizontalFlip).
        Set ``False`` to study memorization/overfitting — augmentation
        suppresses the very degeneration these regimes are meant to induce.
    seed : int
        RNG seed used for deterministic perturbations and subsampling.
    """
    _validate_range("label_noise", label_noise, 0.0, 1.0)
    _validate_range("class_imbalance", class_imbalance, 0.0, 1.0)
    _validate_range("train_fraction", train_fraction, 0.0, 1.0)

    use_pin_memory = pin_memory and torch.cuda.is_available()

    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(_MEAN, _STD),
    ])
    if augment:
        train_transform = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(_MEAN, _STD),
        ])
    else:
        # No augmentation — let the model memorize the (small) train set.
        train_transform = val_transform

    train_set = torchvision.datasets.CIFAR10(
        root=data_root,
        train=True,
        download=True,
        transform=train_transform,
    )
    val_set = torchvision.datasets.CIFAR10(
        root=data_root,
        train=False,
        download=True,
        transform=val_transform,
    )

    # Apply label noise in-place on the base CIFAR targets list.
    train_targets = [int(t) for t in train_set.targets]
    train_set.targets = _apply_label_noise(
        train_targets,
        noise_prob=label_noise,
        n_classes=10,
        seed=seed,
    )

    # Build optional subset for class imbalance / reduced dataset size.
    subset_indices = _build_subset_indices(
        train_set.targets,
        class_imbalance=class_imbalance,
        imbalance_classes=imbalance_classes,
        train_fraction=train_fraction,
        seed=seed,
    )

    train_data = train_set
    if len(subset_indices) != len(train_set):
        train_data = Subset(train_set, subset_indices)

    train_loader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )
    val_loader = DataLoader(
        val_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    return train_loader, val_loader

### 1.5 `train` — train/eval epoch loops

In [ ]:
from __future__ import annotations

"""
src/train.py
============
Training and evaluation loop utilities.

Functions
---------
train_epoch  : Run one full training epoch, return average loss.
eval_epoch   : Run one full validation epoch (no_grad), return loss + accuracy.
"""


import torch
import torch.nn as nn
from tqdm import tqdm


def train_epoch(
    model: nn.Module,
    loader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    show_progress: bool = True,
) -> float:
    """
    Train the model for one epoch.

    Parameters
    ----------
    model     : The network being trained.
    loader    : DataLoader for the training set.
    optimizer : Optimiser (e.g. SGD, Adam).
    criterion : Loss function (e.g. nn.CrossEntropyLoss()).
                Passed in so it is constructed once per run, not per epoch.
    device    : ``torch.device("cuda")`` or ``torch.device("cpu")``.

    Returns
    -------
    float
        Mean training loss over the epoch.
    """
    model.train()
    total_loss = 0.0
    total_samples = 0

    for inputs, labels in tqdm(loader, desc="  train", leave=False, disable=not show_progress):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        if not torch.isfinite(loss):
            print(f"[train_epoch] non-finite loss {loss.item():.4f} — skipping batch")
            optimizer.zero_grad(set_to_none=True)
            continue

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / max(1, total_samples)


@torch.no_grad()
def eval_epoch(
    model: nn.Module,
    loader: torch.utils.data.DataLoader,
    criterion: nn.Module,
    device: torch.device,
    show_progress: bool = True,
) -> tuple[float, float]:
    """
    Evaluate the model for one epoch without gradient tracking.

    Parameters
    ----------
    model     : The network to evaluate.
    loader    : DataLoader for the validation / test set.
    criterion : Loss function - same instance as used in training.
    device    : ``torch.device("cuda")`` or ``torch.device("cpu")``.

    Returns
    -------
    tuple[float, float]
        ``(mean_loss, accuracy)`` where accuracy is in [0, 1].
    """
    model.eval()
    total_loss    = 0.0
    correct       = 0
    total_samples = 0

    for inputs, labels in tqdm(loader, desc="  eval ", leave=False, disable=not show_progress):
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss    = criterion(outputs, labels)

        batch_size = labels.size(0)
        total_loss    += loss.item() * batch_size
        preds          = outputs.argmax(dim=1)
        correct       += preds.eq(labels).sum().item()
        total_samples += batch_size

    mean_loss = total_loss / max(1, total_samples)
    accuracy  = correct / total_samples
    return mean_loss, accuracy

### 1.6 `signals` — the 6 hook signals + SignalLogger

In [ ]:
from __future__ import annotations

"""
src/signals.py
==============
PyTorch hook infrastructure for extracting internal model signals during
training. Metrics are recorded per forward/backward pass and averaged
at epoch end by ``SignalLogger``.

Metrics
-------
- Representation effective rank  (replaces softmax entropy — theoretically grounded)
- Gradient diversity
- Feature reuse
- Neuron sparsity
- Representational isotropy      (replaces redundant embedding variance)
- Activation scale

Usage
-----
    logger = SignalLogger(model, target_layers=["conv2", "fc1"])
    for epoch in range(n_epochs):
        logger.reset()
        train_epoch(model, ...)
        signals = logger.get_epoch_signals()
    logger.remove_hooks()

Signal keys returned by get_epoch_signals()
-------------------------------------------
Each key has the form ``{layer_name}{suffix}`` where suffix is one of:

    _representation_entropy   effective rank of the activation matrix
    _feature_reuse            mean off-diagonal cosine similarity (Gram)
    _gradient_diversity       variance of output gradients across the batch
    _neuron_sparsity          fraction of near-zero activations (adaptive threshold)
    _representational_isotropy  uniformity of per-dimension variance
    _activation_scale         global RMS scale of activations
"""


import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# Module-level canonical suffix constants
# ---------------------------------------------------------------------------

CANONICAL_METRIC_SUFFIXES: tuple[str, ...] = (
    "_representation_entropy",
    "_feature_reuse",
    "_gradient_diversity",
    "_neuron_sparsity",
    "_representational_isotropy",
    "_activation_scale",
)


# ---------------------------------------------------------------------------
# Shared tensor helpers
# ---------------------------------------------------------------------------

def _flatten_batch(x: torch.Tensor) -> torch.Tensor:
    """Flatten all non-batch dimensions into one feature axis."""
    return x.reshape(x.size(0), -1)


def _pool_to_embeddings(x: torch.Tensor) -> torch.Tensor:
    """
    Convert activations to a [B, D] embedding tensor.
    Conv activations are globally averaged over spatial dimensions.
    """
    if x.dim() <= 2:
        return x
    reduce_dims = tuple(range(2, x.dim()))
    return x.mean(dim=reduce_dims)


# ---------------------------------------------------------------------------
# Hook implementations
# ---------------------------------------------------------------------------

class RepresentationEntropyHook:
    """
    Forward hook measuring the effective rank of the activation matrix.

    Effective rank (Roy & Vetterli, 2007) is defined as:

        R_eff(A) = exp( H( σ / Σσ ) )

    where σ are the singular values of the mean-centred activation matrix
    A ∈ R^{B × D}.  The result lies in [1, min(B, D)]:

    - Close to 1  → representations collapsed onto ~1 dimension.
    - Close to D  → representations maximally spread across all dimensions.

    This replaces the previous softmax-entropy formulation, which was
    sensitive to activation *scale* rather than representational *diversity*.
    """

    def __init__(self) -> None:
        self._values: list[float] = []

    @torch.no_grad()
    def __call__(
        self,
        module: nn.Module,
        input: tuple[torch.Tensor, ...],
        output: torch.Tensor,
    ) -> None:
        emb = _pool_to_embeddings(output.detach())  # [B, D]
        B, D = emb.shape[0], emb.shape[1] if emb.dim() > 1 else 1
        if B < 2 or D < 2:
            return

        emb = emb - emb.mean(dim=0, keepdim=True)  # centre columns

        try:
            S = torch.linalg.svdvals(emb)  # [min(B, D)], descending
        except RuntimeError:
            return

        S = S.clamp(min=0.0)
        total = S.sum()
        if total < 1e-9:
            return

        p = S / total
        # Shannon entropy in nats, then exponentiate → effective rank
        H = -(p * torch.log(p + 1e-9)).sum()
        self._values.append(torch.exp(H).item())

    def reset(self) -> None:
        self._values.clear()

    def get_metric(self) -> float:
        return sum(self._values) / len(self._values) if self._values else 0.0


class FeatureReuseDetector:
    """
    Forward hook for average off-diagonal cosine similarity of feature columns.

    Computes a Gram-matrix of centred, L2-normalised feature columns.  The
    mean absolute off-diagonal entry measures how redundant (linearly
    dependent) learned features are — high values indicate collapse to a
    low-diversity feature set.
    """

    def __init__(self) -> None:
        self._scores: list[float] = []

    @torch.no_grad()
    def __call__(
        self,
        module: nn.Module,
        input: tuple[torch.Tensor, ...],
        output: torch.Tensor,
    ) -> None:
        out = _pool_to_embeddings(output.detach())  # [B, D]
        if out.size(0) < 2:
            return

        centered = out - out.mean(dim=0, keepdim=True)
        norms = centered.norm(dim=0, keepdim=True).clamp(min=1e-8)
        normalized = centered / norms  # [B, D], unit-norm columns

        # Gram matrix of cosine similarities between feature columns.
        # Shape [D, D]; diagonal = 1.0 (self-similarity), zeroed out below.
        corr = torch.mm(normalized.T, normalized)
        corr.fill_diagonal_(0.0)
        self._scores.append(corr.abs().mean().item())

    def reset(self) -> None:
        self._scores.clear()

    def get_metric(self) -> float:
        return sum(self._scores) / len(self._scores) if self._scores else 0.0


class GradientDiversityTracker:
    """
    Backward hook measuring variance of output gradients across the batch.

    Low variance indicates all samples produce near-identical gradient
    signals — a sign that the network's training signal has become
    monolithic and uninformative.
    """

    def __init__(self) -> None:
        self._variances: list[float] = []

    def __call__(
        self,
        module: nn.Module,
        grad_input: tuple[torch.Tensor | None, ...],
        grad_output: tuple[torch.Tensor | None, ...],
    ) -> None:
        grads = grad_output[0]
        if grads is None:
            return

        g = grads.detach()
        if g.size(0) == 0:
            return
        if g.dim() > 2:
            g = _flatten_batch(g)

        self._variances.append(g.var(dim=0, unbiased=False).mean().item())

    def reset(self) -> None:
        self._variances.clear()

    def get_metric(self) -> float:
        return sum(self._variances) / len(self._variances) if self._variances else 0.0


class NeuronSparsityTracker:
    """
    Forward hook measuring fraction of near-zero activations.

    Uses an *adaptive* threshold: ``scale * relative_threshold``, where
    ``scale`` is the per-batch mean absolute activation value.  This makes
    the metric invariant to the layer's activation range — a fixed absolute
    threshold (e.g. 1e-3) would report misleadingly high sparsity when
    activations are naturally small-scale, or misleadingly low sparsity for
    large-scale activations.

    Parameters
    ----------
    relative_threshold : float
        Fraction of the mean absolute activation below which a unit is
        considered inactive.  Default 0.01 (1 % of mean scale).
    threshold_mode : str
        ``"adaptive"`` (default) uses relative_threshold × mean|act|.
        ``"absolute"`` uses relative_threshold directly as a fixed cutoff.
    """

    def __init__(
        self,
        relative_threshold: float = 0.01,
        threshold_mode: str = "adaptive",
    ) -> None:
        if threshold_mode not in ("adaptive", "absolute"):
            raise ValueError(f"threshold_mode must be 'adaptive' or 'absolute', got {threshold_mode!r}")
        self.relative_threshold = relative_threshold
        self.threshold_mode = threshold_mode
        self._values: list[float] = []

    @torch.no_grad()
    def __call__(
        self,
        module: nn.Module,
        input: tuple[torch.Tensor, ...],
        output: torch.Tensor,
    ) -> None:
        out = output.detach()
        if out.numel() == 0:
            return

        if self.threshold_mode == "adaptive":
            scale = out.abs().mean()
            threshold = (scale * self.relative_threshold).clamp(min=1e-9)
        else:
            threshold = self.relative_threshold

        sparsity = (out.abs() < threshold).float().mean()
        self._values.append(sparsity.item())

    def reset(self) -> None:
        self._values.clear()

    def get_metric(self) -> float:
        return sum(self._values) / len(self._values) if self._values else 0.0


class RepresentationalIsotropyTracker:
    """
    Forward hook measuring uniformity of per-dimension activation variance.

    Isotropy is defined as:

        isotropy = mean_dim_variance / max_dim_variance

    A value of 1.0 means all dimensions contribute equally (isotropic,
    maximally spread).  A value approaching 0 means variance is concentrated
    in a small number of dimensions — a form of dimensional collapse distinct
    from the scale changes captured by ActivationScaleTracker.

    Replaces the previous EmbeddingVarianceTracker, which measured the same
    underlying quantity as ActivationScaleTracker with a different aggregation.
    """

    def __init__(self) -> None:
        self._values: list[float] = []

    @torch.no_grad()
    def __call__(
        self,
        module: nn.Module,
        input: tuple[torch.Tensor, ...],
        output: torch.Tensor,
    ) -> None:
        emb = _pool_to_embeddings(output.detach())  # [B, D]
        if emb.numel() == 0 or emb.size(0) < 2 or emb.size(1) < 2:
            return

        per_dim_var = emb.var(dim=0, unbiased=False)  # [D]
        max_var = per_dim_var.max()
        if max_var < 1e-9:
            return
        isotropy = per_dim_var.mean() / max_var
        self._values.append(isotropy.item())

    def reset(self) -> None:
        self._values.clear()

    def get_metric(self) -> float:
        return sum(self._values) / len(self._values) if self._values else 0.0


class ActivationScaleTracker:
    """
    Forward hook for global RMS activation scale.

    Tracks whether activation magnitudes are growing or vanishing — a
    precursor to gradient explosion or saturation.  Renamed from
    ActivationVarianceTracker to clarify its role as a scale monitor
    (RMS is more interpretable than raw variance for this purpose).
    """

    def __init__(self) -> None:
        self._values: list[float] = []

    @torch.no_grad()
    def __call__(
        self,
        module: nn.Module,
        input: tuple[torch.Tensor, ...],
        output: torch.Tensor,
    ) -> None:
        out = output.detach()
        if out.numel() == 0:
            return
        rms = out.pow(2).mean().sqrt()
        self._values.append(rms.item())

    def reset(self) -> None:
        self._values.clear()

    def get_metric(self) -> float:
        return sum(self._values) / len(self._values) if self._values else 0.0


# ---------------------------------------------------------------------------
# Signal logger
# ---------------------------------------------------------------------------

class SignalLogger:
    """
    Register and manage all signal hooks on specified model layers.

    Parameters
    ----------
    model : nn.Module
        Model to instrument.
    target_layers : list[str]
        Layer names from ``model.named_modules()``.
    """

    def __init__(self, model: nn.Module, target_layers: list[str]) -> None:
        self.model = model
        self.target_layers = target_layers

        self._handles: list[torch.utils.hooks.RemovableHandle] = []
        self._entropy: dict[str, RepresentationEntropyHook] = {}
        self._reuse: dict[str, FeatureReuseDetector] = {}
        self._grad: dict[str, GradientDiversityTracker] = {}
        self._sparsity: dict[str, NeuronSparsityTracker] = {}
        self._isotropy: dict[str, RepresentationalIsotropyTracker] = {}
        self._scale: dict[str, ActivationScaleTracker] = {}

        self._registered: set[str] = set()
        self._register_hooks()

        missing = set(target_layers) - self._registered
        if missing:
            print(f"[SignalLogger] WARNING: layers not found in model: {sorted(missing)}")

    def _register_hooks(self) -> None:
        for name, module in self.model.named_modules():
            if name not in self.target_layers:
                continue

            e_hook = RepresentationEntropyHook()
            r_hook = FeatureReuseDetector()
            g_hook = GradientDiversityTracker()
            s_hook = NeuronSparsityTracker()
            i_hook = RepresentationalIsotropyTracker()
            a_hook = ActivationScaleTracker()

            self._handles += [
                module.register_forward_hook(e_hook),
                module.register_forward_hook(r_hook),
                module.register_full_backward_hook(g_hook),
                module.register_forward_hook(s_hook),
                module.register_forward_hook(i_hook),
                module.register_forward_hook(a_hook),
            ]

            self._entropy[name] = e_hook
            self._reuse[name] = r_hook
            self._grad[name] = g_hook
            self._sparsity[name] = s_hook
            self._isotropy[name] = i_hook
            self._scale[name] = a_hook
            self._registered.add(name)

    def reset(self) -> None:
        """Clear accumulated values at epoch boundary."""
        for name in self._registered:
            self._entropy[name].reset()
            self._reuse[name].reset()
            self._grad[name].reset()
            self._sparsity[name].reset()
            self._isotropy[name].reset()
            self._scale[name].reset()

    def get_epoch_signals(self) -> dict[str, float]:
        """
        Return averaged epoch signals as a flat dict.

        Keys (one set per tracked layer):
        - ``{layer}_representation_entropy``   effective rank ∈ [1, min(B,D)]
        - ``{layer}_feature_reuse``            mean off-diagonal Gram similarity
        - ``{layer}_gradient_diversity``       mean per-dim gradient variance
        - ``{layer}_neuron_sparsity``          fraction of near-zero activations
        - ``{layer}_representational_isotropy``  mean/max per-dim variance ratio
        - ``{layer}_activation_scale``         global RMS activation magnitude
        """
        signals: dict[str, float] = {}
        for name in self._registered:
            signals[f"{name}_representation_entropy"] = self._entropy[name].get_metric()
            signals[f"{name}_feature_reuse"]          = self._reuse[name].get_metric()
            signals[f"{name}_gradient_diversity"]     = self._grad[name].get_metric()
            signals[f"{name}_neuron_sparsity"]        = self._sparsity[name].get_metric()
            signals[f"{name}_representational_isotropy"] = self._isotropy[name].get_metric()
            signals[f"{name}_activation_scale"]       = self._scale[name].get_metric()
        return signals

    def remove_hooks(self) -> None:
        """De-register all hooks from the model."""
        for handle in self._handles:
            handle.remove()
        self._handles.clear()

    def summary(self, canonical_only: bool = True) -> str:
        """Return a human-readable snapshot of the current epoch signals."""
        signals = self.get_epoch_signals()
        if canonical_only:
            signals = {
                k: v
                for k, v in signals.items()
                if any(k.endswith(suffix) for suffix in CANONICAL_METRIC_SUFFIXES)
            }

        if not signals:
            return "[SignalLogger] No signals recorded yet."

        lines = ["[SignalLogger] Epoch signals:"]
        for key, value in sorted(signals.items()):
            lines.append(f"  {key:<48s} = {value:.6f}")
        return "\n".join(lines)

### 1.7 `predictor` — sliding windows + RandomForest predictor

In [ ]:
from __future__ import annotations

"""
src/predictor.py
================
Sliding-window feature utilities and lightweight instability predictor.

This module is used in two places:
1) Offline training (scripts/train_predictor.py, run_pipeline.py) to build
   a predictor from prior collected runs.
2) Online monitoring (experiments/baseline__run.py) to print per-epoch
   collapse probability while training is still in progress.

Signal name alignment
---------------------
The canonical metric names used here match the key suffixes produced by
``SignalLogger.get_epoch_signals()`` in src/signals.py.  If you rename a
hook or add a new one, update ``CANONICAL_METRIC_NAMES`` and
``_SUFFIX_TO_CANONICAL`` here accordingly.
"""


from pathlib import Path
import pickle
import warnings
from typing import Mapping, Sequence

import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)


# ---------------------------------------------------------------------------
# Canonical metric schema
# ---------------------------------------------------------------------------

CANONICAL_METRIC_NAMES: tuple[str, ...] = (
    "entropy",
    "gradient_diversity",
    "feature_reuse",
    "neuron_sparsity",
    "representational_isotropy",
    "activation_scale",
)

# Maps the signal key suffix (from SignalLogger) → canonical short name.
# Both the long canonical form (e.g. "_representation_entropy") and the
# short form (e.g. "entropy") are handled by canonical_aggregate_features.
_SUFFIX_TO_CANONICAL: dict[str, str] = {
    "_representation_entropy":      "entropy",
    "_gradient_diversity":          "gradient_diversity",
    "_feature_reuse":               "feature_reuse",
    "_neuron_sparsity":             "neuron_sparsity",
    "_representational_isotropy":   "representational_isotropy",
    "_activation_scale":            "activation_scale",
}

# Schema version — increment when the feature schema changes incompatibly.
# v3 adds the persisted ``decision_threshold``.
_SCHEMA_VERSION = 3


# ---------------------------------------------------------------------------
# Feature aggregation
# ---------------------------------------------------------------------------

def canonical_aggregate_features(signals: Mapping[str, float]) -> dict[str, float]:
    """
    Aggregate per-layer signal keys into a canonical feature vector.

    Averages values across all tracked layers for each metric type.
    Input keys may be either:
    - Short canonical names directly (e.g. ``"entropy"``), or
    - Layer-prefixed canonical suffixes (e.g. ``"conv2_representation_entropy"``).

    Returns a dict with exactly the keys in ``CANONICAL_METRIC_NAMES``.
    Missing metrics default to 0.0.
    """
    buckets: dict[str, list[float]] = {name: [] for name in CANONICAL_METRIC_NAMES}

    for key, value in signals.items():
        # Direct canonical name match (e.g. already aggregated upstream).
        if key in buckets:
            buckets[key].append(float(value))
            continue
        # Layer-prefixed key — strip layer name prefix using suffix matching.
        for suffix, canonical_name in _SUFFIX_TO_CANONICAL.items():
            if key.endswith(suffix):
                buckets[canonical_name].append(float(value))
                break

    return {
        name: (sum(values) / len(values) if values else 0.0)
        for name, values in buckets.items()
    }


# ---------------------------------------------------------------------------
# Sliding-window construction
# ---------------------------------------------------------------------------

def _resolve_feature_keys(
    metrics_sequence: Sequence[Mapping[str, float]],
    feature_keys: Sequence[str] | None,
) -> list[str]:
    if feature_keys is not None:
        return [str(k) for k in feature_keys]
    keys: set[str] = set()
    for m in metrics_sequence:
        keys.update(str(k) for k in m.keys())
    return sorted(keys)


def _flatten_window(
    window_metrics: Sequence[Mapping[str, float]],
    feature_keys: Sequence[str],
) -> list[float]:
    vec: list[float] = []
    for step in window_metrics:
        for key in feature_keys:
            vec.append(float(step.get(key, 0.0)))
    return vec


def create_sliding_windows(
    metrics_sequence: Sequence[Mapping[str, float]],
    *,
    window_size: int = 3,
    forecast_horizon: int = 2,
    instability_epoch: int | None = None,
    feature_keys: Sequence[str] | None = None,
    label_mode: str = "forecast",
    return_feature_keys: bool = False,
):
    """
    Convert epoch-wise metrics into supervised windows.

    Two labelling modes are supported via ``label_mode``:

    - ``"forecast"`` (default): a window ending at epoch ``t`` is positive when
      instability first appears strictly within the next ``forecast_horizon``
      epochs, i.e. in ``(t, t+h]``. The window itself is always pre-collapse.

    - ``"detect"``: a window is positive when the collapse onset falls inside the
      window *or* within the next ``forecast_horizon`` epochs — i.e. ``onset <=
      end + h`` (the window start is guaranteed to be ``<= onset``; see below).
      Windows lying entirely in the collapse *aftermath* (``start > onset``) are
      dropped, because a stuck-at-collapse state is neither a forecast target nor
      a useful "stable" negative — keeping them as negatives inverts the model.
      This is the right mode for collapses with no internal precursor (e.g. an
      exogenous LR spike): the model learns to fire as collapse begins.

    Parameters
    ----------
    metrics_sequence : sequence of dicts
        One dict per epoch.  Keys are metric names; values are floats.
    window_size : int
        Number of past epochs in each feature window.
    forecast_horizon : int
        Number of future epochs to check for the instability label.
    instability_epoch : int | None
        Epoch index (0-based) of first detected instability.  ``None`` means
        the run was stable — all windows are labelled 0.
    feature_keys : sequence of str | None
        Ordered list of keys to extract from each step.  If ``None``, inferred
        from the union of all keys in ``metrics_sequence`` (sorted).
    label_mode : str
        ``"forecast"`` or ``"detect"`` (see above).
    return_feature_keys : bool
        When ``True``, also return the resolved feature key list as the third
        element of the tuple.

    Returns
    -------
    (X, y) or (X, y, keys)
        X : list of flat float vectors, length = n_windows.
        y : list of int labels (0 or 1).
        keys : list[str] — only when ``return_feature_keys=True``.
    """
    if window_size < 1:
        raise ValueError(f"window_size must be >= 1, got {window_size}")
    if forecast_horizon < 1:
        raise ValueError(f"forecast_horizon must be >= 1, got {forecast_horizon}")
    if label_mode not in ("forecast", "detect"):
        raise ValueError(f"label_mode must be 'forecast' or 'detect', got {label_mode!r}")

    if len(metrics_sequence) < window_size + forecast_horizon:
        keys = _resolve_feature_keys(metrics_sequence, feature_keys)
        if return_feature_keys:
            return [], [], keys
        return [], []

    keys = _resolve_feature_keys(metrics_sequence, feature_keys)
    X: list[list[float]] = []
    y: list[int] = []

    max_start = len(metrics_sequence) - window_size - forecast_horizon + 1
    for start in range(max_start):
        end = start + window_size - 1
        future_end = end + forecast_horizon

        if label_mode == "detect":
            # Drop pure-aftermath windows; positive when onset is in-window or
            # within the horizon ahead.
            if instability_epoch is not None and start > instability_epoch:
                continue
            positive = (
                instability_epoch is not None
                and instability_epoch <= future_end
            )
        else:
            positive = (
                instability_epoch is not None
                and end < instability_epoch <= future_end
            )

        window = metrics_sequence[start : start + window_size]
        X.append(_flatten_window(window, keys))
        y.append(int(positive))

    if return_feature_keys:
        return X, y, keys
    return X, y


# ---------------------------------------------------------------------------
# Predictor
# ---------------------------------------------------------------------------

class Predictor:
    """Random-forest predictor wrapper with robust save/load and proba API."""

    def __init__(
        self,
        *,
        window_size: int = 3,
        forecast_horizon: int = 2,
        feature_keys: Sequence[str] | None = None,
        decision_threshold: float = 0.5,
        model=None,
    ) -> None:
        self.window_size = int(window_size)
        self.forecast_horizon = int(forecast_horizon)
        self.feature_keys = list(feature_keys) if feature_keys is not None else None
        # Probability cutoff for the positive (collapse) class. Calibrated by
        # ``tune_threshold``; 0.5 reproduces sklearn's default ``predict``.
        self.decision_threshold = float(decision_threshold)
        self.model = model or RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced_subsample",
            # OOB probabilities give an unbiased basis for threshold tuning —
            # in-sample probabilities saturate because the RF fits train perfectly.
            oob_score=True,
            bootstrap=True,
            n_jobs=-1,
        )

    def train(
        self,
        X: Sequence[Sequence[float]],
        y: Sequence[int],
        *,
        feature_keys: Sequence[str] | None = None,
    ) -> None:
        if not X:
            raise ValueError("Cannot train predictor: X is empty.")
        if len(X) != len(y):
            raise ValueError(f"X/y length mismatch: {len(X)} vs {len(y)}")

        if feature_keys is not None:
            self.feature_keys = [str(k) for k in feature_keys]

        labels = [int(v) for v in y]
        unique = sorted(set(labels))

        if len(unique) == 1:
            # Low-data edge case: all windows belong to one class.
            # Caller should be warned — a DummyClassifier is not a predictor.
            print(
                f"[Predictor] WARNING: only class {unique[0]} in training data. "
                "Falling back to DummyClassifier. Collect more runs with "
                "both stable and unstable examples."
            )
            self.model = DummyClassifier(strategy="constant", constant=unique[0])

        self.model.fit(X, labels)

    def predict(self, X: Sequence[Sequence[float]]) -> np.ndarray:
        """Hard class predictions using the calibrated ``decision_threshold``."""
        return (self.predict_proba(X) >= self.decision_threshold).astype(int)

    def predict_proba(self, X: Sequence[Sequence[float]]) -> np.ndarray:
        """Return probability of class ``1`` (incoming instability)."""
        proba = self.model.predict_proba(X)
        classes = [int(c) for c in getattr(self.model, "classes_", [0, 1])]

        if len(classes) == 1:
            val = 1.0 if classes[0] == 1 else 0.0
            return np.full(len(X), val, dtype=float)

        if 1 not in classes:
            return np.zeros(len(X), dtype=float)

        pos_index = classes.index(1)
        return proba[:, pos_index]

    def _oob_positive_proba(self) -> np.ndarray | None:
        """
        Return out-of-bag P(class==1) for the training rows, or ``None`` when
        OOB is unavailable (e.g. DummyClassifier, ``oob_score=False``).

        Rows where a sample was never out-of-bag are NaN in
        ``oob_decision_function_``; callers must mask those out.
        """
        oob = getattr(self.model, "oob_decision_function_", None)
        if oob is None:
            return None
        classes = [int(c) for c in getattr(self.model, "classes_", [])]
        if 1 not in classes:
            return None
        return np.asarray(oob)[:, classes.index(1)]

    def tune_threshold(
        self,
        X: Sequence[Sequence[float]],
        y: Sequence[int],
        *,
        metric: str = "f1",
    ) -> float:
        """
        Calibrate ``decision_threshold`` to maximise ``metric`` (default F1).

        Uses out-of-bag probabilities when available (unbiased) and falls back
        to in-sample ``predict_proba`` otherwise. Thresholds with no positives in
        ``y`` — or where the model produced a single class — leave the default
        0.5 in place. Returns the chosen threshold.
        """
        if metric != "f1":
            raise ValueError(f"Unsupported metric: {metric!r} (only 'f1').")

        labels = np.asarray([int(v) for v in y])
        if labels.size == 0 or labels.sum() == 0 or len(set(labels.tolist())) < 2:
            return self.decision_threshold

        probs = self._oob_positive_proba()
        eval_labels = labels
        if probs is not None:
            finite = np.isfinite(probs)
            if finite.sum() >= 2 and labels[finite].sum() > 0:
                probs, eval_labels = probs[finite], labels[finite]
            else:
                probs = None  # too few OOB rows — fall back
        if probs is None:
            probs = self.predict_proba(X)

        candidates = sorted(set(float(p) for p in probs))
        if not candidates:
            return self.decision_threshold

        best_threshold, best_f1 = self.decision_threshold, -1.0
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UndefinedMetricWarning)
            for thr in candidates:
                preds = (probs >= thr).astype(int)
                score = f1_score(eval_labels, preds, zero_division=0)
                if score > best_f1:
                    best_f1, best_threshold = score, thr

        self.decision_threshold = float(best_threshold)
        return self.decision_threshold

    def predict_probability_from_window(
        self,
        window_metrics: Sequence[Mapping[str, float]],
    ) -> float:
        """Compute collapse probability from a raw window of per-epoch signal dicts."""
        if self.feature_keys is None:
            raise ValueError("predictor.feature_keys is not set.")
        if len(window_metrics) != self.window_size:
            raise ValueError(
                f"Expected {self.window_size} steps, got {len(window_metrics)}."
            )
        x = _flatten_window(window_metrics, self.feature_keys)
        return float(self.predict_proba([x])[0])

    def evaluate(
        self,
        X: Sequence[Sequence[float]],
        y: Sequence[int],
    ) -> dict[str, float]:
        preds = self.predict(X)
        probs = self.predict_proba(X)
        acc = float(accuracy_score(y, preds))
        print(f"Accuracy: {acc:.4f}  (decision_threshold={self.decision_threshold:.3f})")
        print(classification_report(y, preds, digits=4, zero_division=0))

        metrics: dict[str, float] = {"accuracy": acc}
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UndefinedMetricWarning)
            try:
                auc = float(roc_auc_score(y, probs))
                print(f"ROC-AUC: {auc:.4f}")
                metrics["roc_auc"] = auc
            except ValueError:
                print("ROC-AUC: undefined (only one class present in y_true).")
        return metrics

    def save(self, path: str | Path) -> Path:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "schema_version": _SCHEMA_VERSION,
            "model": self.model,
            "window_size": self.window_size,
            "forecast_horizon": self.forecast_horizon,
            "feature_keys": self.feature_keys,
            "decision_threshold": self.decision_threshold,
        }
        with path.open("wb") as f:
            pickle.dump(payload, f)
        return path

    @classmethod
    def load(cls, path: str | Path) -> "Predictor":
        path = Path(path)
        with path.open("rb") as f:
            payload = pickle.load(f)

        saved_version = payload.get("schema_version", 1)
        if saved_version != _SCHEMA_VERSION:
            print(
                f"[Predictor] WARNING: saved schema version {saved_version} "
                f"!= current {_SCHEMA_VERSION}. Feature keys may be stale. "
                "Re-train the predictor with the current codebase."
            )

        return cls(
            window_size=int(payload["window_size"]),
            forecast_horizon=int(payload["forecast_horizon"]),
            feature_keys=payload.get("feature_keys"),
            decision_threshold=float(payload.get("decision_threshold", 0.5)),
            model=payload["model"],
        )

### 1.8 `labeller` — instability detection from val-accuracy

In [ ]:
from __future__ import annotations

"""
src/labeller.py
===============
Objective instability detection for training runs.

Detection criterion:
- Start checking after ``burn_in`` epochs.
- Track running peak validation accuracy.
- If accuracy drops by more than ``drop_threshold`` from the latest peak
  within ``window`` epochs, and this condition persists for at least
  ``sustain_epochs`` consecutive epochs, mark the run unstable.

All public functions share their core logic via the private
``_scan_instability`` helper to avoid code duplication.
"""



def _scan_instability(
    val_accuracies: list[float],
    *,
    drop_threshold: float,
    window: int,
    burn_in: int,
    sustain_epochs: int,
    chance_level: float | None = None,
    chance_tolerance: float = 0.02,
    chance_burn_in: int = 2,
) -> int | None:
    """
    Scan ``val_accuracies`` and return the epoch index of first detected
    instability, or ``None`` if no instability is found.

    Two failure modes are detected; the earliest detection wins:

    1. **Drop from peak** — accuracy falls more than ``drop_threshold`` below the
       running peak within ``window`` epochs, sustained for ``sustain_epochs``.
    2. **Stuck at chance** (only when ``chance_level`` is set) — accuracy stays
       at or below ``chance_level + chance_tolerance`` for ``sustain_epochs``
       consecutive epochs from ``chance_burn_in`` onward. This catches runs that
       never learn (e.g. a divergent learning rate) or collapse immediately, which
       the drop-from-peak rule misses because they have no peak to fall from.

    This is the single implementation shared by ``is_unstable`` and
    ``get_instability_epoch`` — keep them in sync by editing here only.
    """
    drop_epoch = _scan_drop_from_peak(
        val_accuracies,
        drop_threshold=drop_threshold,
        window=window,
        burn_in=burn_in,
        sustain_epochs=sustain_epochs,
    )
    chance_epoch = _scan_stuck_at_chance(
        val_accuracies,
        chance_level=chance_level,
        chance_tolerance=chance_tolerance,
        chance_burn_in=chance_burn_in,
        sustain_epochs=sustain_epochs,
    )

    candidates = [e for e in (drop_epoch, chance_epoch) if e is not None]
    return min(candidates) if candidates else None


def _scan_drop_from_peak(
    val_accuracies: list[float],
    *,
    drop_threshold: float,
    window: int,
    burn_in: int,
    sustain_epochs: int,
) -> int | None:
    """Detect a sustained drop below the running peak (see ``_scan_instability``)."""
    if len(val_accuracies) <= burn_in:
        return None

    running_peak_val = val_accuracies[0]
    running_peak_idx = 0
    consecutive_violations = 0

    for i, acc in enumerate(val_accuracies):
        # Update peak — use latest occurrence to anchor window to recent peak.
        if acc >= running_peak_val:
            running_peak_val = acc
            running_peak_idx = i

        if i <= burn_in:
            continue

        drop = running_peak_val - acc
        epochs_since_peak = i - running_peak_idx
        violating = drop > drop_threshold and epochs_since_peak <= window

        if violating:
            consecutive_violations += 1
            if consecutive_violations >= sustain_epochs:
                return i
        else:
            consecutive_violations = 0

    return None


def _scan_stuck_at_chance(
    val_accuracies: list[float],
    *,
    chance_level: float | None,
    chance_tolerance: float,
    chance_burn_in: int,
    sustain_epochs: int,
) -> int | None:
    """
    Detect a run pinned at chance accuracy (see ``_scan_instability``).

    Returns the first epoch of the sustained at-chance run, or ``None``.
    """
    if chance_level is None:
        return None

    ceiling = chance_level + chance_tolerance
    consecutive = 0
    for i, acc in enumerate(val_accuracies):
        if i < chance_burn_in:
            continue
        if acc <= ceiling:
            consecutive += 1
            if consecutive >= sustain_epochs:
                return i - sustain_epochs + 1  # first epoch of the at-chance run
        else:
            consecutive = 0

    return None


def is_unstable(
    val_accuracies: list[float],
    *,
    drop_threshold: float = 0.08,
    window: int = 5,
    burn_in: int = 10,
    sustain_epochs: int = 1,
    chance_level: float | None = None,
    chance_tolerance: float = 0.02,
    chance_burn_in: int = 2,
) -> bool:
    """
    Return ``True`` if the run exhibits instability / degeneration.

    Parameters
    ----------
    val_accuracies : list[float]
        Validation accuracy per epoch in [0, 1].
    drop_threshold : float
        Minimum drop below running peak to count as a violation.
    window : int
        Maximum epochs since most recent peak to qualify the drop.
    burn_in : int
        Ignore early epochs before this index.
    sustain_epochs : int
        Number of consecutive violating epochs required for detection.
    chance_level : float | None
        If set, also flag runs stuck at or below ``chance_level +
        chance_tolerance`` accuracy (e.g. 0.10 for 10-class CIFAR). ``None``
        disables this check (drop-from-peak only).
    chance_tolerance : float
        Margin above ``chance_level`` still considered "at chance".
    chance_burn_in : int
        First epoch index at which the at-chance check becomes active.
    """
    if sustain_epochs < 1:
        raise ValueError(f"sustain_epochs must be >= 1, got {sustain_epochs}")
    return _scan_instability(
        val_accuracies,
        drop_threshold=drop_threshold,
        window=window,
        burn_in=burn_in,
        sustain_epochs=sustain_epochs,
        chance_level=chance_level,
        chance_tolerance=chance_tolerance,
        chance_burn_in=chance_burn_in,
    ) is not None


def get_instability_epoch(
    val_accuracies: list[float],
    *,
    drop_threshold: float = 0.08,
    window: int = 5,
    burn_in: int = 10,
    sustain_epochs: int = 1,
    chance_level: float | None = None,
    chance_tolerance: float = 0.02,
    chance_burn_in: int = 2,
) -> int | None:
    """
    Return epoch index where instability is first detected, or ``None``.

    See :func:`is_unstable` for the ``chance_level`` parameters.
    """
    if sustain_epochs < 1:
        raise ValueError(f"sustain_epochs must be >= 1, got {sustain_epochs}")
    return _scan_instability(
        val_accuracies,
        drop_threshold=drop_threshold,
        window=window,
        burn_in=burn_in,
        sustain_epochs=sustain_epochs,
        chance_level=chance_level,
        chance_tolerance=chance_tolerance,
        chance_burn_in=chance_burn_in,
    )


def label_run(
    val_accuracies: list[float],
    **kwargs,
) -> dict[str, bool | int | None]:
    """
    Convenience wrapper returning both binary label and first detection epoch.
    """
    epoch = get_instability_epoch(val_accuracies, **kwargs)
    return {"unstable": epoch is not None, "instability_epoch": epoch}

### 1.9 `evaluation` — LORO-CV + heuristic baselines

In [ ]:
from __future__ import annotations

"""
src/evaluation.py
=================
Leave-one-run-out cross-validation (LORO-CV) and baseline comparisons
for the instability predictor.

Design
------
- ``RunData``              dataclass holding per-run metrics + ground-truth
- ``leave_one_run_out_cv`` trains and evaluates the RF predictor via LORO-CV
- ``evaluate_baselines``   runs heuristic baselines on the same LORO splits
- ``MajorityClassBaseline``, ``ValAccDropBaseline``, ``RandomBaseline``
  produce window-level predictions without using learned signal features

Typical usage
-------------

    runs = [RunData(run_id, metrics_seq, val_accs, instability_epoch), ...]
    cv_result = leave_one_run_out_cv(runs, window_size=3, forecast_horizon=2)
    base_result = evaluate_baselines(runs, window_size=3, forecast_horizon=2)
"""


import math
import warnings
from dataclasses import dataclass, field
from typing import Any, Callable, Sequence

import numpy as np
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)



# ---------------------------------------------------------------------------
# RunData
# ---------------------------------------------------------------------------

@dataclass
class RunData:
    """
    Container for one training run's per-epoch signals and ground-truth label.

    Parameters
    ----------
    run_id : str
        Unique identifier (e.g. ``"delayed_collapse__seed100"``).
    metrics_sequence : list[dict[str, float]]
        One signal dict per epoch, produced by ``SignalLogger.get_epoch_signals()``.
    val_accuracies : list[float]
        Validation accuracy per epoch, parallel to ``metrics_sequence``.
    instability_epoch : int | None
        0-based epoch index of first detected instability.  ``None`` → stable run.
    """
    run_id: str
    metrics_sequence: list[dict[str, float]]
    val_accuracies: list[float]
    instability_epoch: int | None = field(default=None)


# ---------------------------------------------------------------------------
# Metric helpers
# ---------------------------------------------------------------------------

def _compute_metrics(
    y_true: list[int],
    y_pred: list[int],
    y_prob: list[float] | None,
) -> dict[str, float]:
    result: dict[str, float] = {
        "accuracy":  float(accuracy_score(y_true, y_pred)),
        "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
    }
    if y_prob is not None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UndefinedMetricWarning)
            try:
                result["roc_auc"] = float(roc_auc_score(y_true, y_prob))
            except ValueError:
                result["roc_auc"] = float("nan")
    else:
        result["roc_auc"] = float("nan")
    return result


def _aggregate_folds(fold_metrics: list[dict[str, float]]) -> dict[str, float]:
    """Return mean ± std across folds for each metric key."""
    if not fold_metrics:
        return {}
    agg: dict[str, float] = {}
    for key in fold_metrics[0]:
        # Skip non-numeric columns (e.g. run_id) — only aggregate real numbers.
        first = fold_metrics[0][key]
        if isinstance(first, bool) or not isinstance(first, (int, float)):
            continue
        vals = [m[key] for m in fold_metrics if not math.isnan(m[key])]
        if vals:
            agg[f"{key}_mean"] = float(np.mean(vals))
            agg[f"{key}_std"]  = float(np.std(vals))
        else:
            agg[f"{key}_mean"] = float("nan")
            agg[f"{key}_std"]  = float("nan")
    return agg


# ---------------------------------------------------------------------------
# LORO-CV
# ---------------------------------------------------------------------------

def leave_one_run_out_cv(
    runs: list[RunData],
    *,
    window_size: int = 3,
    forecast_horizon: int = 2,
    label_mode: str = "detect",
    predictor_kwargs: dict[str, Any] | None = None,
    aggregate_fn: Callable[[dict[str, float]], dict[str, float]] = canonical_aggregate_features,
    verbose: bool = True,
) -> dict[str, Any]:
    """
    Leave-one-run-out cross-validation for the instability predictor.

    For each run ``r``:
    - Train a fresh ``Predictor`` on windows from all other runs.
    - Evaluate on windows from ``r`` (held-out).

    Parameters
    ----------
    runs : list[RunData]
        At least 2 runs; needs at least one stable and one unstable for
        meaningful ROC-AUC.
    window_size, forecast_horizon : int
        Passed to ``create_sliding_windows``.
    predictor_kwargs : dict | None
        Extra kwargs forwarded to ``Predictor.__init__``.
    aggregate_fn : callable
        Feature aggregation function applied to each epoch's signal dict
        before windowing.  Defaults to ``canonical_aggregate_features``.
    verbose : bool
        Print per-fold summary to stdout.

    Returns
    -------
    dict with keys:
        ``fold_results``   — list of per-fold dicts (run_id, n_windows, metrics)
        ``aggregate``      — mean ± std across folds for each metric
        ``skipped_folds``  — run_ids where the held-out fold had no windows
        ``class_counts``   — {"positive": n, "negative": n} across all folds
    """
    if len(runs) < 2:
        raise ValueError(f"LORO-CV requires at least 2 runs, got {len(runs)}")

    predictor_kwargs = predictor_kwargs or {}
    fold_results: list[dict[str, Any]] = []
    skipped: list[str] = []
    all_y_true: list[int] = []
    all_y_pred: list[int] = []

    # Pre-aggregate features so we do it once per run.
    agg_sequences: list[list[dict[str, float]]] = [
        [aggregate_fn(ep) for ep in run.metrics_sequence]
        for run in runs
    ]

    # Resolve a shared, stable feature key set from ALL runs (union).
    all_keys: set[str] = set()
    for seq in agg_sequences:
        for ep in seq:
            all_keys.update(ep.keys())
    feature_keys = sorted(all_keys)

    if verbose:
        print(f"[LORO-CV] {len(runs)} runs  |  feature_keys={feature_keys}")

    for hold_idx, held_run in enumerate(runs):
        # Build training windows from all other runs.
        X_train: list[list[float]] = []
        y_train: list[int] = []

        for i, run in enumerate(runs):
            if i == hold_idx:
                continue
            X_i, y_i = create_sliding_windows(
                agg_sequences[i],
                window_size=window_size,
                forecast_horizon=forecast_horizon,
                instability_epoch=run.instability_epoch,
                feature_keys=feature_keys,
                label_mode=label_mode,
            )
            X_train.extend(X_i)
            y_train.extend(y_i)

        if not X_train:
            if verbose:
                print(f"  [SKIP] {held_run.run_id}: training set has no windows")
            skipped.append(held_run.run_id)
            continue

        predictor = Predictor(
            window_size=window_size,
            forecast_horizon=forecast_horizon,
            feature_keys=feature_keys,
            **predictor_kwargs,
        )
        predictor.train(X_train, y_train)
        # Calibrate the decision threshold on the training fold (uses unbiased
        # OOB probabilities) so held-out hard predictions reflect a usable
        # operating point rather than the saturated 0.5 default.
        predictor.tune_threshold(X_train, y_train)

        # Evaluate on held-out run.
        X_test, y_test = create_sliding_windows(
            agg_sequences[hold_idx],
            window_size=window_size,
            forecast_horizon=forecast_horizon,
            instability_epoch=held_run.instability_epoch,
            feature_keys=feature_keys,
            label_mode=label_mode,
        )

        if not X_test:
            if verbose:
                print(f"  [SKIP] {held_run.run_id}: held-out run too short for windows")
            skipped.append(held_run.run_id)
            continue

        y_pred = list(predictor.predict(X_test).tolist())
        y_prob = list(predictor.predict_proba(X_test).tolist())
        metrics = _compute_metrics(y_test, y_pred, y_prob)

        all_y_true.extend(y_test)
        all_y_pred.extend(y_pred)

        fold_results.append({
            "run_id":    held_run.run_id,
            "n_windows": len(y_test),
            "n_pos":     sum(y_test),
            **metrics,
        })

        if verbose:
            inst = held_run.instability_epoch
            print(
                f"  Fold {hold_idx+1:2d}/{len(runs)} "
                f"| {held_run.run_id:<40s} "
                f"| inst={inst!s:<6} "
                f"| n={len(y_test):3d} "
                f"| acc={metrics['accuracy']:.3f} "
                f"| f1={metrics['f1']:.3f} "
                f"| auc={metrics.get('roc_auc', float('nan')):.3f}"
            )

    aggregate = _aggregate_folds([f for f in fold_results])
    class_counts = {
        "positive": sum(all_y_true),
        "negative": len(all_y_true) - sum(all_y_true),
    }

    if verbose and fold_results:
        print(f"\n[LORO-CV] Aggregate over {len(fold_results)} folds:")
        for key, val in sorted(aggregate.items()):
            print(f"  {key:<28s} = {val:.4f}")
        print(f"  class_counts: {class_counts}")

    return {
        "fold_results":  fold_results,
        "aggregate":     aggregate,
        "skipped_folds": skipped,
        "class_counts":  class_counts,
    }


# ---------------------------------------------------------------------------
# Heuristic baselines
# ---------------------------------------------------------------------------

class MajorityClassBaseline:
    """
    Always predicts the majority class from the training windows.

    Calibrated per fold (majority label in training split).
    """

    def __init__(self) -> None:
        self._majority: int = 0

    def fit(self, y_train: Sequence[int]) -> None:
        if not y_train:
            self._majority = 0
            return
        self._majority = 1 if sum(y_train) * 2 >= len(y_train) else 0

    def predict(self, n: int) -> list[int]:
        return [self._majority] * n

    def predict_proba(self, n: int) -> list[float]:
        return [float(self._majority)] * n


class ValAccDropBaseline:
    """
    Heuristic: predict instability when the val accuracy in the window has
    dropped more than ``drop_threshold`` from the window's peak.

    This simulates what a practitioner would do without signal features —
    watch for accuracy degradation and raise an alert.  It does NOT use any
    internal model signals, only the validation accuracy sequence.
    """

    def __init__(self, drop_threshold: float = 0.05) -> None:
        self.drop_threshold = drop_threshold

    def predict_windows(
        self,
        val_accuracies: list[float],
        *,
        window_size: int,
        forecast_horizon: int,
        instability_epoch: int | None = None,
        label_mode: str = "forecast",
    ) -> list[int]:
        """
        Produce a predicted label for each sliding window position.

        Positive (1) when: ``peak_in_window − last_in_window > drop_threshold``.

        ``instability_epoch``/``label_mode`` mirror :func:`create_sliding_windows`
        so the produced predictions stay aligned with the predictor's windows
        (in ``"detect"`` mode the pure-aftermath windows are skipped identically).
        """
        n = len(val_accuracies)
        max_start = n - window_size - forecast_horizon + 1
        preds: list[int] = []
        for start in range(max_start):
            if (
                label_mode == "detect"
                and instability_epoch is not None
                and start > instability_epoch
            ):
                continue
            window_vals = val_accuracies[start : start + window_size]
            peak = max(window_vals)
            last = window_vals[-1]
            preds.append(int(peak - last > self.drop_threshold))
        return preds


class RandomBaseline:
    """
    Predicts class 1 at the training-set positive rate (random with calibrated prior).

    Useful for verifying that any classifier scores above chance.
    """

    def __init__(self, seed: int = 0) -> None:
        self._rng = np.random.default_rng(seed)
        self._pos_rate: float = 0.5

    def fit(self, y_train: Sequence[int]) -> None:
        if not y_train:
            self._pos_rate = 0.5
            return
        self._pos_rate = sum(y_train) / len(y_train)

    def predict(self, n: int) -> list[int]:
        return [int(v) for v in (self._rng.random(n) < self._pos_rate)]

    def predict_proba(self, n: int) -> list[float]:
        return [float(self._pos_rate)] * n


# ---------------------------------------------------------------------------
# Baseline LORO evaluation
# ---------------------------------------------------------------------------

def evaluate_baselines(
    runs: list[RunData],
    *,
    window_size: int = 3,
    forecast_horizon: int = 2,
    label_mode: str = "detect",
    val_acc_drop_threshold: float = 0.05,
    verbose: bool = True,
) -> dict[str, dict[str, Any]]:
    """
    Evaluate all heuristic baselines using the same LORO splits.

    Returns a dict mapping baseline name → aggregate LORO metrics.

    Parameters
    ----------
    runs : list[RunData]
        Same list passed to ``leave_one_run_out_cv``.
    window_size, forecast_horizon : int
        Must match the values used in the predictor LORO evaluation.
    label_mode : str
        Must match the value used in ``leave_one_run_out_cv`` so baseline windows
        align with the predictor's. See :func:`create_sliding_windows`.
    val_acc_drop_threshold : float
        Drop threshold for ``ValAccDropBaseline``.
    verbose : bool
        Print per-baseline aggregate metrics.
    """
    baselines: dict[str, Any] = {
        "majority_class": MajorityClassBaseline(),
        f"val_acc_drop_{val_acc_drop_threshold}": ValAccDropBaseline(val_acc_drop_threshold),
        "random": RandomBaseline(seed=0),
    }

    results: dict[str, dict[str, Any]] = {}

    for name, baseline in baselines.items():
        fold_metrics: list[dict[str, float]] = []

        for hold_idx, held_run in enumerate(runs):
            # Build ground-truth windows for held-out run.
            # We only need the y labels — use a dummy 1-feature sequence.
            dummy_seq = [{"_": 0.0}] * len(held_run.metrics_sequence)
            _, y_test = create_sliding_windows(
                dummy_seq,
                window_size=window_size,
                forecast_horizon=forecast_horizon,
                instability_epoch=held_run.instability_epoch,
                feature_keys=["_"],
                label_mode=label_mode,
            )
            if not y_test:
                continue

            if isinstance(baseline, ValAccDropBaseline):
                y_pred = baseline.predict_windows(
                    held_run.val_accuracies,
                    window_size=window_size,
                    forecast_horizon=forecast_horizon,
                    instability_epoch=held_run.instability_epoch,
                    label_mode=label_mode,
                )
                y_prob = [float(p) for p in y_pred]
            else:
                # Fit majority/random on training labels.
                train_y: list[int] = []
                for i, run in enumerate(runs):
                    if i == hold_idx:
                        continue
                    dummy_i = [{"_": 0.0}] * len(run.metrics_sequence)
                    _, y_i = create_sliding_windows(
                        dummy_i,
                        window_size=window_size,
                        forecast_horizon=forecast_horizon,
                        instability_epoch=run.instability_epoch,
                        feature_keys=["_"],
                        label_mode=label_mode,
                    )
                    train_y.extend(y_i)

                baseline.fit(train_y)
                y_pred = baseline.predict(len(y_test))
                y_prob = baseline.predict_proba(len(y_test))

            metrics = _compute_metrics(y_test, y_pred, y_prob)
            fold_metrics.append(metrics)

        agg = _aggregate_folds(fold_metrics)
        results[name] = {"fold_metrics": fold_metrics, "aggregate": agg}

        if verbose:
            print(f"[Baseline] {name}:")
            for key, val in sorted(agg.items()):
                print(f"  {key:<28s} = {val:.4f}")

    return results


# ---------------------------------------------------------------------------
# Forecasting: lead-time (horizon) sweep
# ---------------------------------------------------------------------------

def forecast_horizon_sweep(
    runs: list[RunData],
    *,
    horizons: Sequence[int] = (1, 2, 3, 5, 8),
    window_size: int = 3,
    label_mode: str = "forecast",
    val_acc_drop_threshold: float = 0.05,
    predictor_kwargs: dict[str, Any] | None = None,
    verbose: bool = True,
) -> dict[int, dict[str, Any]]:
    """
    Measure forecasting skill as a function of lead time (``forecast_horizon``).

    For each horizon ``h`` this runs the full LORO-CV predictor evaluation and
    the ``val_acc_drop`` baseline at that horizon (``label_mode="forecast"`` by
    default — positive windows are strictly *before* collapse onset). The result
    answers the research question: how many epochs ahead can internal signals
    forecast collapse, and do they beat simply watching validation accuracy?

    Returns
    -------
    dict[int, dict] keyed by horizon, each with:
        ``predictor``    — aggregate LORO metrics for the RF predictor
        ``val_acc_drop`` — aggregate LORO metrics for the val-accuracy baseline
        ``class_counts`` — {"positive": n, "negative": n} across folds
    """
    results: dict[int, dict[str, Any]] = {}
    base_key = f"val_acc_drop_{val_acc_drop_threshold}"

    for h in horizons:
        cv = leave_one_run_out_cv(
            runs,
            window_size=window_size,
            forecast_horizon=h,
            label_mode=label_mode,
            predictor_kwargs=predictor_kwargs,
            verbose=False,
        )
        base = evaluate_baselines(
            runs,
            window_size=window_size,
            forecast_horizon=h,
            label_mode=label_mode,
            val_acc_drop_threshold=val_acc_drop_threshold,
            verbose=False,
        )
        pred_agg = cv["aggregate"]
        base_agg = base.get(base_key, {}).get("aggregate", {})
        results[int(h)] = {
            "predictor": pred_agg,
            "val_acc_drop": base_agg,
            "class_counts": cv["class_counts"],
        }

        if verbose:
            def _g(agg: dict[str, float], key: str) -> float:
                return agg.get(key, float("nan"))
            print(
                f"  h={h:>2} | "
                f"predictor roc_auc={_g(pred_agg, 'roc_auc_mean'):.3f} "
                f"f1={_g(pred_agg, 'f1_mean'):.3f} "
                f"recall={_g(pred_agg, 'recall_mean'):.3f} "
                f"|| val_acc_drop roc_auc={_g(base_agg, 'roc_auc_mean'):.3f} "
                f"f1={_g(base_agg, 'f1_mean'):.3f} "
                f"| pos_windows={cv['class_counts']['positive']}"
            )

    return results


# ---------------------------------------------------------------------------
# Convenience: print comparison table
# ---------------------------------------------------------------------------

def print_comparison_table(
    predictor_result: dict[str, Any],
    baseline_results: dict[str, dict[str, Any]],
    metrics: Sequence[str] = ("accuracy_mean", "f1_mean", "roc_auc_mean"),
) -> None:
    """
    Print a side-by-side comparison of predictor vs. baselines.

    Parameters
    ----------
    predictor_result : dict
        Return value of ``leave_one_run_out_cv``.
    baseline_results : dict
        Return value of ``evaluate_baselines``.
    metrics : sequence of str
        Aggregate metric keys to display.
    """
    all_rows: list[tuple[str, dict[str, float]]] = [
        ("RF Predictor (LORO-CV)", predictor_result["aggregate"]),
    ]
    for name, res in baseline_results.items():
        all_rows.append((f"  Baseline: {name}", res["aggregate"]))

    col_w = 35
    header = f"{'Model':<{col_w}}" + "".join(f"{m:>18}" for m in metrics)
    print("\n" + "=" * len(header))
    print(header)
    print("-" * len(header))
    for label, agg in all_rows:
        row = f"{label:<{col_w}}"
        for m in metrics:
            val = agg.get(m, float("nan"))
            row += f"  {val:>7.4f}      " if not math.isnan(val) else f"  {'—':>7}      "
        print(row)
    print("=" * len(header))

### 1.10 `signal_lag` — signal lead-time analysis

In [ ]:
from __future__ import annotations

"""
analysis/signal_lag.py
======================
Temporal precedence analysis: measure how many epochs BEFORE detected
instability each internal signal shows statistically significant deviation
from its stable-run baseline.

If signals change BEFORE collapse, the predictor can actually forecast ahead.
If they only change AT or AFTER collapse, the system has no predictive value
beyond detecting an already-happening event.

Usage
-----
As a module (import into notebooks or scripts):

    results = compute_signal_lags(run_csv_path="output/instability_runs/session_xxx/runs/")
    print_lag_table(results)

As a CLI:

    python -m analysis.signal_lag --session-dir output/instability_runs/session_xxx
    python -m analysis.signal_lag --run-csv path/to/run.csv

Output
------
For each signal metric, returns:
- ``lead_epochs``: median epochs before collapse at which signal crossed
  the deviation threshold (positive = precedes collapse)
- ``detection_rate``: fraction of unstable runs where signal showed
  pre-collapse deviation
- ``stable_mean``, ``stable_std``: distribution in stable runs (for normalisation)
"""


import argparse
import sys
from pathlib import Path
from typing import Any

import numpy as np

# Signal metric suffixes from src/signals.py (duplicated here to avoid
# importing torch when this module is used as a pure analysis script).
_CANONICAL_SUFFIXES = (
    "_representation_entropy",
    "_feature_reuse",
    "_gradient_diversity",
    "_neuron_sparsity",
    "_representational_isotropy",
    "_activation_scale",
)

_LOOKBACK = 15  # epochs before instability to examine


# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------

def _load_csv(path: Path) -> list[dict[str, Any]]:
    """Read a CSV into a list of row dicts (values as strings)."""
    import csv
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def _parse_float(value: str) -> float | None:
    try:
        return float(value)
    except (ValueError, TypeError):
        return None


def _load_runs_from_dir(runs_dir: Path) -> list[dict[str, Any]]:
    """
    Load all per-run CSVs from a session runs/ directory.
    Returns a list of run dicts with keys: run_id, regime, unstable,
    instability_epoch, rows (list of epoch row dicts).
    """
    run_files = sorted(runs_dir.glob("*.csv"))
    if not run_files:
        raise FileNotFoundError(f"No CSV files found in {runs_dir}")

    runs: list[dict[str, Any]] = []
    for csv_path in run_files:
        rows = _load_csv(csv_path)
        if not rows:
            continue

        last = rows[-1]
        inst_raw = last.get("instability_epoch", "")
        instability_epoch = int(float(inst_raw)) if inst_raw not in ("", "None", None) else None
        unstable_raw = last.get("unstable", "False")
        unstable = str(unstable_raw).strip().lower() in ("true", "1", "yes")

        runs.append({
            "run_id":            last.get("run_id", csv_path.stem),
            "regime":            last.get("regime", ""),
            "unstable":          unstable,
            "instability_epoch": instability_epoch,
            "rows":              rows,
        })
    return runs


def _load_single_csv(csv_path: Path) -> list[dict[str, Any]]:
    """
    Load a summary-style CSV where each row is one epoch of one run.
    Expects columns: run_id, epoch, unstable, instability_epoch, plus signal columns.
    """
    rows = _load_csv(csv_path)

    # Group by run_id.
    by_run: dict[str, list[dict[str, Any]]] = {}
    for row in rows:
        rid = row.get("run_id", "unknown")
        by_run.setdefault(rid, []).append(row)

    runs: list[dict[str, Any]] = []
    for rid, run_rows in by_run.items():
        run_rows.sort(key=lambda r: int(r.get("epoch", 0)))
        last = run_rows[-1]
        inst_raw = last.get("instability_epoch", "")
        instability_epoch = int(float(inst_raw)) if inst_raw not in ("", "None", None) else None
        unstable_raw = last.get("unstable", "False")
        unstable = str(unstable_raw).strip().lower() in ("true", "1", "yes")

        runs.append({
            "run_id":            rid,
            "regime":            last.get("regime", ""),
            "unstable":          unstable,
            "instability_epoch": instability_epoch,
            "rows":              run_rows,
        })
    return runs


# ---------------------------------------------------------------------------
# Signal extraction helpers
# ---------------------------------------------------------------------------

def _extract_signal_series(
    rows: list[dict[str, Any]],
    signal_key: str,
) -> list[float | None]:
    """Return the value of ``signal_key`` for each epoch row (None if missing/invalid)."""
    return [_parse_float(row.get(signal_key, "")) for row in rows]


def _discover_signal_keys(runs: list[dict[str, Any]]) -> list[str]:
    """Find all signal columns present in at least one run."""
    keys: set[str] = set()
    for run in runs:
        for row in run["rows"][:1]:  # headers are consistent — check first row
            for k in row.keys():
                if any(k.endswith(suffix) for suffix in _CANONICAL_SUFFIXES):
                    keys.add(k)
    return sorted(keys)


# ---------------------------------------------------------------------------
# Core analysis
# ---------------------------------------------------------------------------

def compute_signal_lags(
    *,
    session_dir: str | Path | None = None,
    run_csv: str | Path | None = None,
    lookback: int = _LOOKBACK,
    deviation_threshold_z: float = 2.0,
) -> dict[str, dict[str, Any]]:
    """
    Compute per-signal lead times relative to detected instability.

    Parameters
    ----------
    session_dir : Path | None
        Path to a session directory containing a ``runs/`` subdirectory
        (output of ``run_many_regimes.py``).
    run_csv : Path | None
        Path to a single summary CSV with ``run_id`` column.
        Exactly one of ``session_dir`` or ``run_csv`` must be provided.
    lookback : int
        Number of epochs before instability to examine.
    deviation_threshold_z : float
        Z-score threshold (relative to stable-run mean/std) at which a
        signal is considered "anomalous" in a given epoch.

    Returns
    -------
    dict mapping signal_key → {
        "lead_epochs": float,       median epochs before collapse where anomaly first appears
        "detection_rate": float,    fraction of unstable runs with a pre-collapse anomaly
        "stable_mean": float,
        "stable_std": float,
        "unstable_pre_mean": float, mean over lookback window in unstable runs
    }
    """
    if (session_dir is None) == (run_csv is None):
        raise ValueError("Provide exactly one of session_dir or run_csv.")

    if session_dir is not None:
        runs_dir = Path(session_dir) / "runs"
        runs = _load_runs_from_dir(runs_dir)
    else:
        runs = _load_single_csv(Path(run_csv))  # type: ignore[arg-type]

    signal_keys = _discover_signal_keys(runs)
    if not signal_keys:
        raise ValueError(
            "No canonical signal columns found in the provided data. "
            "Make sure the CSVs were produced by run_many_regimes.py."
        )

    stable_runs   = [r for r in runs if not r["unstable"]]
    unstable_runs = [r for r in runs if r["unstable"] and r["instability_epoch"] is not None]

    if not stable_runs:
        print("[WARNING] No stable runs found — cannot compute baseline distribution.")
    if not unstable_runs:
        print("[WARNING] No unstable runs found — nothing to analyse.")
        return {}

    results: dict[str, dict[str, Any]] = {}

    for sig_key in signal_keys:
        # Stable baseline.
        stable_vals: list[float] = []
        for run in stable_runs:
            series = _extract_signal_series(run["rows"], sig_key)
            stable_vals.extend(v for v in series if v is not None)

        stable_mean = float(np.mean(stable_vals)) if stable_vals else 0.0
        stable_std  = float(np.std(stable_vals))  if stable_vals else 1.0
        if stable_std < 1e-9:
            stable_std = 1.0  # avoid division by zero

        # Per-unstable-run analysis.
        lead_times: list[int] = []
        pre_means:  list[float] = []

        for run in unstable_runs:
            inst_ep = run["instability_epoch"]  # epoch index (0-based)
            rows = run["rows"]
            series = _extract_signal_series(rows, sig_key)

            # Slice the lookback window: [inst_ep - lookback, inst_ep).
            start = max(0, inst_ep - lookback)
            pre_series = [v for v in series[start:inst_ep] if v is not None]

            if not pre_series:
                continue

            pre_means.append(float(np.mean(pre_series)))

            # Find the earliest epoch in the lookback window where the
            # z-score exceeds the threshold.
            anomaly_offset: int | None = None
            for offset, val in enumerate(pre_series):
                z = abs(val - stable_mean) / stable_std
                if z > deviation_threshold_z:
                    anomaly_offset = offset
                    break  # first anomaly in the window

            if anomaly_offset is not None:
                # lead_time = epochs before instability where anomaly appeared
                window_len = len(pre_series)
                lead_times.append(window_len - anomaly_offset)

        results[sig_key] = {
            "lead_epochs":       float(np.median(lead_times)) if lead_times else float("nan"),
            "detection_rate":    len(lead_times) / len(unstable_runs) if unstable_runs else float("nan"),
            "stable_mean":       stable_mean,
            "stable_std":        stable_std,
            "unstable_pre_mean": float(np.mean(pre_means)) if pre_means else float("nan"),
            "n_unstable_runs":   len(unstable_runs),
            "n_stable_runs":     len(stable_runs),
        }

    return results


# ---------------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------------

def print_lag_table(results: dict[str, dict[str, Any]]) -> None:
    """Print a formatted lead-time table sorted by lead_epochs descending."""
    if not results:
        print("[signal_lag] No results to display.")
        return

    # Sort: most predictive (highest lead_epochs & detection_rate) first.
    sorted_keys = sorted(
        results,
        key=lambda k: (
            results[k].get("lead_epochs") or 0.0,
            results[k].get("detection_rate") or 0.0,
        ),
        reverse=True,
    )

    col = 52
    header = (
        f"{'Signal':<{col}}"
        f"{'Lead (epochs)':>14}"
        f"{'Det. Rate':>12}"
        f"{'Stable μ':>12}"
        f"{'Stable σ':>12}"
        f"{'PreCrash μ':>12}"
    )
    print("\n" + "=" * len(header))
    print("Signal Lead-Time Analysis (epochs BEFORE detected collapse)")
    print("=" * len(header))
    print(header)
    print("-" * len(header))

    for key in sorted_keys:
        r = results[key]
        lead = r.get("lead_epochs", float("nan"))
        det  = r.get("detection_rate", float("nan"))
        sm   = r.get("stable_mean", float("nan"))
        ss   = r.get("stable_std", float("nan"))
        pm   = r.get("unstable_pre_mean", float("nan"))

        lead_str = f"{lead:.1f}" if not (isinstance(lead, float) and lead != lead) else "—"
        det_str  = f"{det:.2f}"  if not (isinstance(det,  float) and det  != det)  else "—"

        print(
            f"{key:<{col}}"
            f"{lead_str:>14}"
            f"{det_str:>12}"
            f"{sm:>12.4f}"
            f"{ss:>12.4f}"
            f"{pm:>12.4f}"
        )

    print("=" * len(header))
    first = results[sorted_keys[0]]
    print(
        f"\nNote: analysis over {first['n_unstable_runs']} unstable runs, "
        f"{first['n_stable_runs']} stable runs."
    )


# ---------------------------------------------------------------------------
# CLI entry point
# ---------------------------------------------------------------------------

def _build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Compute signal lead-times relative to instability onset."
    )
    group = parser.add_mutually_exclusive_group(required=True)
    group.add_argument(
        "--session-dir", type=str, default=None,
        help="Path to a session directory (contains runs/ subdir).",
    )
    group.add_argument(
        "--run-csv", type=str, default=None,
        help="Path to a single per-epoch CSV with run_id column.",
    )
    parser.add_argument(
        "--lookback", type=int, default=_LOOKBACK,
        help=f"Epochs before instability to examine (default {_LOOKBACK}).",
    )
    parser.add_argument(
        "--z-threshold", type=float, default=2.0,
        help="Z-score threshold for anomaly detection (default 2.0).",
    )
    return parser

### 1.11 `plots` — visualisation helpers

In [ ]:
from __future__ import annotations

"""
analysis/plots.py
=================
Visualization utilities for instability experiment results.

All functions return the matplotlib Figure object so callers can save or
display it.  No interactive display is shown unless the caller calls
``plt.show()`` or ``fig.show()``.

Functions
---------
- ``plot_val_accuracy_curves``  — per-run val accuracy with instability markers
- ``plot_signal_trajectories``  — one or more signal metrics over epochs
- ``plot_lead_time_bar``        — lead-time results from signal_lag analysis
- ``plot_cv_metrics``           — per-fold LORO-CV metrics as a bar chart
- ``plot_feature_importance``   — RF feature importance from a trained Predictor
"""


from pathlib import Path
from typing import Any, Sequence

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np


# ---------------------------------------------------------------------------
# Colour palette
# ---------------------------------------------------------------------------

_REGIME_COLOURS: dict[str, str] = {
    "normal":               "#4CAF50",
    "label_noise":          "#F44336",
    "over_regularization":  "#FF9800",
    "high_learning_rate":   "#E91E63",
    "overtraining":         "#9C27B0",
    "class_imbalance":      "#2196F3",
    "reduced_dataset_size": "#00BCD4",
    "delayed_collapse":     "#FF5722",
    "warm_then_overfit":    "#795548",
}
_DEFAULT_STABLE_COLOUR   = "#4CAF50"
_DEFAULT_UNSTABLE_COLOUR = "#F44336"
_INSTABILITY_MARKER_COLOUR = "black"


def _regime_colour(regime: str, stable: bool) -> str:
    base = _REGIME_COLOURS.get(regime, "#607D8B")
    return base if not stable else _DEFAULT_STABLE_COLOUR


# ---------------------------------------------------------------------------
# Val accuracy curves
# ---------------------------------------------------------------------------

def plot_val_accuracy_curves(
    runs: list[dict[str, Any]],
    *,
    title: str = "Validation Accuracy over Training",
    figsize: tuple[float, float] = (10, 5),
    alpha: float = 0.8,
    mark_instability: bool = True,
) -> plt.Figure:
    """
    Plot validation accuracy curves for multiple runs.

    Parameters
    ----------
    runs : list of dicts with keys:
        ``run_id`` (str), ``val_accuracies`` (list[float]),
        ``instability_epoch`` (int | None), ``regime`` (str).
    mark_instability : bool
        If True, draw a vertical dashed line at ``instability_epoch`` for
        unstable runs.
    """
    fig, ax = plt.subplots(figsize=figsize)

    for run in runs:
        accs = run["val_accuracies"]
        epochs = list(range(1, len(accs) + 1))
        inst = run.get("instability_epoch")
        regime = run.get("regime", "")
        stable = inst is None
        colour = _regime_colour(regime, stable)
        label = run.get("run_id", "")

        ax.plot(epochs, [a * 100 for a in accs],
                color=colour, alpha=alpha, linewidth=1.2, label=label)

        if mark_instability and inst is not None:
            ax.axvline(x=inst + 1, color=colour, linestyle="--", alpha=0.5, linewidth=0.8)

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation Accuracy (%)")
    ax.set_title(title)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
    ax.grid(True, alpha=0.3)
    if len(runs) <= 12:
        ax.legend(fontsize=7, loc="lower right")
    fig.tight_layout()
    return fig


# ---------------------------------------------------------------------------
# Signal trajectories
# ---------------------------------------------------------------------------

def plot_signal_trajectories(
    epoch_signals: list[dict[str, float]],
    signal_keys: Sequence[str],
    *,
    instability_epoch: int | None = None,
    run_id: str = "",
    figsize: tuple[float, float] | None = None,
) -> plt.Figure:
    """
    Plot one or more signal metrics over epochs for a single run.

    Parameters
    ----------
    epoch_signals : list of dicts
        One dict per epoch from ``SignalLogger.get_epoch_signals()``.
    signal_keys : sequence of str
        Which keys to plot (e.g. ``["conv2_representation_entropy", ...]``).
    instability_epoch : int | None
        0-based epoch index at which instability was detected.  A vertical
        marker is drawn if provided.
    """
    n = len(signal_keys)
    if figsize is None:
        figsize = (10, 2.5 * n)

    fig, axes = plt.subplots(n, 1, figsize=figsize, sharex=True)
    if n == 1:
        axes = [axes]

    epochs = list(range(1, len(epoch_signals) + 1))

    for ax, key in zip(axes, signal_keys):
        values = [ep.get(key, float("nan")) for ep in epoch_signals]
        ax.plot(epochs, values, color="#1976D2", linewidth=1.3)
        ax.set_ylabel(key, fontsize=8)
        ax.grid(True, alpha=0.3)

        if instability_epoch is not None:
            ax.axvline(
                x=instability_epoch + 1,
                color=_INSTABILITY_MARKER_COLOUR,
                linestyle="--",
                linewidth=1.0,
                label="instability",
            )

    axes[-1].set_xlabel("Epoch")
    title = f"Signal Trajectories — {run_id}" if run_id else "Signal Trajectories"
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    return fig


# ---------------------------------------------------------------------------
# Lead-time bar chart
# ---------------------------------------------------------------------------

def plot_lead_time_bar(
    lag_results: dict[str, dict[str, Any]],
    *,
    title: str = "Signal Lead-Time Before Collapse",
    figsize: tuple[float, float] | None = None,
    min_detection_rate: float = 0.0,
) -> plt.Figure:
    """
    Bar chart of median lead epochs for each signal, from signal_lag analysis.

    Parameters
    ----------
    lag_results : dict
        Return value of ``analysis.signal_lag.compute_signal_lags``.
    min_detection_rate : float
        Only show signals with detection rate >= this value.
    """
    import math

    filtered = {
        k: v for k, v in lag_results.items()
        if not math.isnan(v.get("lead_epochs", float("nan")))
        and v.get("detection_rate", 0.0) >= min_detection_rate
    }

    if not filtered:
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.text(0.5, 0.5, "No data to display", ha="center", va="center",
                transform=ax.transAxes)
        return fig

    keys = sorted(filtered, key=lambda k: filtered[k]["lead_epochs"], reverse=True)
    leads = [filtered[k]["lead_epochs"] for k in keys]
    rates = [filtered[k]["detection_rate"] for k in keys]

    n = len(keys)
    if figsize is None:
        figsize = (10, max(4, n * 0.5))

    fig, ax = plt.subplots(figsize=figsize)
    colours = [
        "#1976D2" if r >= 0.7 else "#64B5F6" if r >= 0.4 else "#BBDEFB"
        for r in rates
    ]
    bars = ax.barh(range(n), leads, color=colours, edgecolor="white")
    ax.set_yticks(range(n))
    ax.set_yticklabels([k.replace("_representation_entropy", "_eff_rank") for k in keys],
                       fontsize=8)
    ax.set_xlabel("Median Lead (epochs before collapse)")
    ax.set_title(title)
    ax.axvline(x=0, color="gray", linewidth=0.5)
    ax.grid(True, axis="x", alpha=0.3)

    # Annotate bars with detection rate.
    for i, (bar, rate) in enumerate(zip(bars, rates)):
        ax.text(
            bar.get_width() + 0.05, i,
            f"{rate:.0%}",
            va="center", fontsize=7, color="#555",
        )

    fig.tight_layout()
    return fig


# ---------------------------------------------------------------------------
# LORO-CV per-fold metrics
# ---------------------------------------------------------------------------

def plot_cv_metrics(
    cv_result: dict[str, Any],
    *,
    metrics: Sequence[str] = ("accuracy", "f1", "roc_auc"),
    title: str = "LORO-CV Fold Metrics",
    figsize: tuple[float, float] | None = None,
) -> plt.Figure:
    """
    Bar chart of per-fold metrics from ``leave_one_run_out_cv``.

    Parameters
    ----------
    cv_result : dict
        Return value of ``src.evaluation.leave_one_run_out_cv``.
    metrics : sequence of str
        Which metric columns to plot.
    """
    fold_results = cv_result.get("fold_results", [])
    if not fold_results:
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.text(0.5, 0.5, "No fold results", ha="center", va="center",
                transform=ax.transAxes)
        return fig

    run_ids = [f["run_id"] for f in fold_results]
    n_folds = len(run_ids)
    n_metrics = len(metrics)
    if figsize is None:
        figsize = (max(10, n_folds * 0.6), 4 * n_metrics)

    fig, axes = plt.subplots(n_metrics, 1, figsize=figsize, sharex=True)
    if n_metrics == 1:
        axes = [axes]

    colours = ["#1976D2", "#388E3C", "#F57C00", "#C62828"]
    x = np.arange(n_folds)

    for ax, metric, colour in zip(axes, metrics, colours * 10):
        vals = [f.get(metric, float("nan")) for f in fold_results]
        ax.bar(x, vals, color=colour, alpha=0.8, edgecolor="white")

        # Mean line.
        agg = cv_result.get("aggregate", {})
        mean_val = agg.get(f"{metric}_mean")
        if mean_val is not None:
            ax.axhline(mean_val, color="black", linestyle="--", linewidth=1.0,
                       label=f"mean={mean_val:.3f}")
            ax.legend(fontsize=8)

        ax.set_ylabel(metric)
        ax.set_ylim(0, 1.05)
        ax.grid(True, axis="y", alpha=0.3)

    axes[-1].set_xticks(x)
    axes[-1].set_xticklabels(run_ids, rotation=45, ha="right", fontsize=7)
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    return fig


# ---------------------------------------------------------------------------
# Feature importance
# ---------------------------------------------------------------------------

def plot_feature_importance(
    predictor: Any,
    *,
    top_n: int = 20,
    title: str = "RF Feature Importance",
    figsize: tuple[float, float] | None = None,
) -> plt.Figure:
    """
    Horizontal bar chart of the trained RF predictor's feature importances.

    Parameters
    ----------
    predictor : Predictor
        A trained ``src.predictor.Predictor`` instance.
    top_n : int
        Show only the top N most important features.
    """
    rf = predictor.model
    feature_keys = predictor.feature_keys or []

    if not hasattr(rf, "feature_importances_"):
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.text(0.5, 0.5, "Model has no feature_importances_ (non-RF?)",
                ha="center", va="center", transform=ax.transAxes)
        return fig

    importances: np.ndarray = rf.feature_importances_
    n_features = len(importances)
    window_size = int(getattr(predictor, "window_size", 1) or 1)

    if feature_keys and len(feature_keys) == n_features:
        labels = list(feature_keys)
    elif feature_keys and len(feature_keys) * window_size == n_features:
        # Windowed features are flattened step-outer, key-inner (see
        # predictor._flatten_window): index = step * n_keys + k. Label each with
        # its metric and how many epochs back in the window it came from.
        n_keys = len(feature_keys)
        labels = [
            f"{feature_keys[i % n_keys]} @t-{window_size - 1 - (i // n_keys)}"
            for i in range(n_features)
        ]
    else:
        labels = [f"feature_{i}" for i in range(n_features)]

    # Top N.
    indices = np.argsort(importances)[::-1][:top_n]
    top_importances = importances[indices]
    top_labels = [labels[i] for i in indices]

    n = len(top_labels)
    if figsize is None:
        figsize = (10, max(4, n * 0.4))

    fig, ax = plt.subplots(figsize=figsize)
    colours = plt.cm.Blues(np.linspace(0.4, 0.9, n))[::-1]
    ax.barh(range(n), top_importances[::-1], color=colours[::-1], edgecolor="white")
    ax.set_yticks(range(n))
    ax.set_yticklabels(top_labels[::-1], fontsize=8)
    ax.set_xlabel("Mean Decrease in Impurity")
    ax.set_title(title)
    ax.grid(True, axis="x", alpha=0.3)
    fig.tight_layout()
    return fig


# ---------------------------------------------------------------------------
# Forecasting horizon sweep
# ---------------------------------------------------------------------------

def plot_horizon_sweep(
    sweep_result: dict[int, dict[str, Any]],
    *,
    metric: str = "roc_auc",
    title: str | None = None,
    figsize: tuple[float, float] = (8, 5),
) -> plt.Figure:
    """
    Plot forecasting skill vs lead time: RF predictor (internal signals) against
    the val-accuracy-drop baseline, over forecast horizons.

    Parameters
    ----------
    sweep_result : dict
        Return value of ``src.evaluation.forecast_horizon_sweep``.
    metric : str
        Base metric name, e.g. ``"roc_auc"``, ``"f1"``, ``"recall"`` — the
        ``"{metric}_mean"``/``"{metric}_std"`` keys are read from each aggregate.
    """
    horizons = sorted(sweep_result.keys())
    mean_key, std_key = f"{metric}_mean", f"{metric}_std"

    def _series(side: str, key: str) -> list[float]:
        return [sweep_result[h][side].get(key, float("nan")) for h in horizons]

    pred_mean = _series("predictor", mean_key)
    pred_std = _series("predictor", std_key)
    base_mean = _series("val_acc_drop", mean_key)

    fig, ax = plt.subplots(figsize=figsize)
    ax.errorbar(
        horizons, pred_mean, yerr=pred_std, marker="o", capsize=3,
        color="#1976D2", linewidth=1.6, label="RF predictor (internal signals)",
    )
    ax.plot(
        horizons, base_mean, marker="s", linestyle="--",
        color="#F57C00", linewidth=1.4, label="val-accuracy-drop baseline",
    )
    if metric == "roc_auc":
        ax.axhline(0.5, color="gray", linewidth=0.8, linestyle=":", label="chance")

    ax.set_xlabel("Forecast horizon (epochs ahead of collapse onset)")
    ax.set_ylabel(metric)
    ax.set_title(title or f"Forecasting skill vs lead time ({metric})")
    ax.set_xticks(horizons)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig


# ---------------------------------------------------------------------------
# Convenience: save all figures to a directory
# ---------------------------------------------------------------------------

def save_figure(fig: plt.Figure, path: str | Path, dpi: int = 150) -> Path:
    """Save a matplotlib Figure and return its path."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return path

## 2. Runtime configuration

In [ ]:
%matplotlib inline
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))


## 3. Quick smoke test

Trains one healthy run and one run that collapses mid-training (a 50x LR spike at
epoch 6) on a 10% CIFAR-10 subsample, then trains a Random-Forest predictor on their
signals. The LR spike creates a genuine healthy->collapse transition so both classes are
present. **In-sample only** — a fast end-to-end check that the pipeline executes.
(Adapted from `run_pipeline.py`.)

In [ ]:
import csv
from pathlib import Path
import torch.nn as nn

def quick_smoke_test(epochs=10, batch_size=128, seed=42,
                     window_size=3, forecast_horizon=2,
                     output_dir='output'):
    seed_everything(seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    def run_one(regime_name, label_noise=0.0, lr_spike=None):
        # lr_spike=(epoch_index, factor): multiply LR mid-run to force a real
        # healthy->collapse transition (a forecastable event, unlike label noise
        # which is merely bad from the start).
        print(f'\n=> Regime: {regime_name}  (device={DEVICE})')
        train_loader, val_loader = get_cifar_loaders(
            batch_size=batch_size, label_noise=label_noise, train_fraction=0.1)
        model = SimpleCNN().to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()
        logger = SignalLogger(model, target_layers=['conv2', 'fc1'])
        metrics_history, val_acc_history = [], []
        for epoch in range(epochs):
            if lr_spike is not None and epoch == lr_spike[0]:
                for pg in optimizer.param_groups:
                    pg['lr'] *= lr_spike[1]
                print(f'  [LR spike] epoch {epoch+1}: lr x{lr_spike[1]}')
            logger.reset()
            train_loss = train_epoch(model, train_loader, optimizer, criterion,
                                     DEVICE, show_progress=False)
            signals = logger.get_epoch_signals()
            _, val_acc = eval_epoch(model, val_loader, criterion, DEVICE,
                                    show_progress=False)
            print(f'  Epoch {epoch+1:02d}/{epochs} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}')
            metrics_history.append(signals)
            val_acc_history.append(val_acc)
        logger.remove_hooks()
        return metrics_history, val_acc_history

    print('--- Phase 1: Data Generation ---')
    healthy_metrics, healthy_acc = run_one('Healthy')
    unstable_metrics, unstable_acc = run_one('Unstable', lr_spike=(6, 50.0))

    print('\n--- Phase 2: Instability Labelling ---')
    healthy_inst = get_instability_epoch(healthy_acc, drop_threshold=0.02,
                                         burn_in=2, chance_level=0.10)
    unstable_inst = get_instability_epoch(unstable_acc, drop_threshold=0.02,
                                          burn_in=2, chance_level=0.10)
    print(f'  Healthy  instability epoch: {healthy_inst}')
    print(f'  Unstable instability epoch: {unstable_inst}')

    print('\n--- Phase 3: Train Random Forest Predictor ---')
    feat_h = [canonical_aggregate_features(m) for m in healthy_metrics]
    feat_u = [canonical_aggregate_features(m) for m in unstable_metrics]
    X_h, y_h, feature_keys = create_sliding_windows(
        feat_h, window_size=window_size, forecast_horizon=forecast_horizon,
        instability_epoch=healthy_inst, label_mode='detect', return_feature_keys=True)
    X_u, y_u = create_sliding_windows(
        feat_u, window_size=window_size, forecast_horizon=forecast_horizon,
        instability_epoch=unstable_inst, feature_keys=feature_keys, label_mode='detect')
    X_train, y_train = X_h + X_u, y_h + y_u
    if not X_train:
        print('Not enough epochs to create sliding windows.'); return None
    predictor = Predictor(window_size=window_size, forecast_horizon=forecast_horizon,
                          feature_keys=feature_keys)
    predictor.train(X_train, y_train)
    saved = predictor.save(output_dir / 'predictor' / 'random_forest.pkl')
    print(f'[INFO] Predictor saved -> {saved}')
    print(f'\n[EVAL] In-sample evaluation on {len(X_train)} windows (NOT held-out):')
    predictor.evaluate(X_train, y_train)
    print('\nSmoke test complete.')
    return predictor

_ = quick_smoke_test()


## 4. Batch experiments across regimes

Runs each regime across several seeds, logs per-epoch signals to CSV, and returns
`RunData` objects in memory for evaluation. Adapted from `experiments/run_many_regimes.py`.

Defaults are a Colab-friendly subset. For the full study set
`regimes=tuple(ALL_REGIMES)` and more seeds (slower).

In [ ]:
import csv, json
from datetime import datetime
from pathlib import Path
import torch.nn as nn

def _canonical_signals(signals):
    return {k: v for k, v in signals.items()
            if any(k.endswith(s) for s in CANONICAL_METRIC_SUFFIXES)}

def _write_csv(path, rows):
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)

def run_batch_experiments(
    regimes=('normal', 'high_learning_rate', 'delayed_collapse'),
    seeds=(100, 101),
    epochs_override=None,
    model_name='simple',
    batch_size=128,
    data_root='./data',
    output_root='./output/instability_runs',
    drop_threshold=0.08, drop_window=5, burn_in=10, sustain_epochs=3,
    chance_level=0.10,
):
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    session_dir = Path(output_root) / f'session_{stamp}'
    runs_dir = session_dir / 'runs'
    runs_dir.mkdir(parents=True, exist_ok=True)

    print('=' * 90)
    print(f'Batch runner | session={session_dir}')
    print(f'regimes={list(regimes)}  seeds={list(seeds)}  '
          f'default_model={model_name}  device={DEVICE}')
    print('=' * 90)

    run_data_list, summaries = [], []
    total = len(regimes) * len(seeds)
    idx = 0
    for regime in regimes:
        cfg = get_regime_config(regime)
        epochs = epochs_override if epochs_override is not None else cfg.epochs
        rmodel = cfg.model or model_name
        target_layers = resolve_target_layers(rmodel, None)
        for seed in seeds:
            idx += 1
            run_id = f'{regime}__seed{seed}'
            print(f'[{idx:03d}/{total:03d}] {run_id}  epochs={epochs}  lr={cfg.lr:.5f}  '
                  f'model={rmodel}  augment={cfg.augment}  frac={cfg.train_fraction}')
            seed_everything(seed)
            train_loader, val_loader = get_cifar_loaders(
                batch_size=batch_size, data_root=data_root,
                label_noise=cfg.label_noise, class_imbalance=cfg.class_imbalance,
                train_fraction=cfg.train_fraction, augment=cfg.augment, seed=seed)
            model = build_model(rmodel).to(DEVICE)
            logger = SignalLogger(model, target_layers=target_layers)
            optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr,
                                         weight_decay=cfg.weight_decay)
            criterion = nn.CrossEntropyLoss()
            val_history, rows, metrics_sequence = [], [], []
            try:
                for epoch in range(1, epochs + 1):
                    if cfg.lr_boost_at_epoch is not None and epoch == cfg.lr_boost_at_epoch:
                        for pg in optimizer.param_groups:
                            pg['lr'] *= cfg.lr_boost_factor
                    logger.reset()
                    train_loss = train_epoch(model, train_loader, optimizer, criterion,
                                             DEVICE, show_progress=False)
                    signals = logger.get_epoch_signals()
                    val_loss, val_acc = eval_epoch(model, val_loader, criterion,
                                                   DEVICE, show_progress=False)
                    val_history.append(val_acc)
                    collapse = label_run(val_history, drop_threshold=drop_threshold,
                                         window=drop_window, burn_in=burn_in,
                                         sustain_epochs=sustain_epochs,
                                         chance_level=chance_level)
                    canon = _canonical_signals(signals)
                    metrics_sequence.append(canon)
                    row = {'run_id': run_id, 'regime': regime, 'seed': seed,
                           'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                           'val_acc': val_acc, 'peak_val_acc': max(val_history),
                           'unstable': bool(collapse['unstable']),
                           'instability_epoch': collapse['instability_epoch']}
                    row.update(canon)
                    rows.append(row)
            finally:
                logger.remove_hooks()
            final = label_run(val_history, drop_threshold=drop_threshold,
                              window=drop_window, burn_in=burn_in,
                              sustain_epochs=sustain_epochs,
                              chance_level=chance_level)
            _write_csv(runs_dir / f'{run_id}.csv', rows)
            run_data_list.append(RunData(run_id=run_id, metrics_sequence=metrics_sequence,
                                         val_accuracies=val_history,
                                         instability_epoch=final['instability_epoch']))
            summaries.append({'run_id': run_id, 'regime': regime, 'seed': seed,
                              'epochs': epochs, 'best_val_acc': max(val_history),
                              'final_val_acc': val_history[-1],
                              'unstable': bool(final['unstable']),
                              'instability_epoch': final['instability_epoch']})
            print(f'    done  best={max(val_history)*100:.2f}%  '
                  f"final={val_history[-1]*100:.2f}%  unstable={final['unstable']}")
    _write_csv(session_dir / 'run_summary.csv', summaries)
    (session_dir / 'manifest.json').write_text(json.dumps(
        {'session_dir': str(session_dir), 'regimes': list(regimes),
         'seeds': list(seeds), 'default_model': model_name}, indent=2),
        encoding='utf-8')
    print('=' * 90)
    print(f'Session written to: {session_dir}')
    n_unstable = sum(1 for r in run_data_list if r.instability_epoch is not None)
    print(f'Runs: {len(run_data_list)}  ({n_unstable} unstable, '
          f'{len(run_data_list) - n_unstable} stable)')
    return session_dir, run_data_list


In [ ]:
# Run the batch. Bump seeds / regimes / epochs for a fuller study.
SESSION_DIR, RUNS = run_batch_experiments(
    regimes=('normal', 'high_learning_rate', 'delayed_collapse'),
    seeds=(100, 101),
)


## 5. Held-out evaluation (LORO-CV vs. baselines)

Leave-one-run-out cross-validation of the RF predictor, compared against majority-class,
val-accuracy-drop, and random baselines. Adapted from `scripts/evaluate_predictor.py`.

**Framing — early *detection*, not pure forecasting.** A window is positive if the
collapse onset falls inside it or within the next `forecast_horizon` epochs
(`label_mode='detect'`); pure post-collapse windows are dropped. This is the honest
target for these regimes: `delayed_collapse` is triggered by an exogenous LR spike with
no internal precursor, so its pre-collapse signals are identical to a healthy run — the
model can detect collapse as it begins, not predict an unforeseeable shock. Section 6's
lead-time analysis is the complementary 'how early do signals move' view.

Each fold also calibrates its decision threshold on the training runs (using the RF's
out-of-bag probabilities), so `f1`/`precision`/`recall` reflect a usable alarm
operating point — not the saturated 0.5 default. `roc_auc` stays threshold-free.

In [ ]:
if len(RUNS) < 2:
    print('Need >= 2 runs for LORO-CV. Re-run section 4 with more seeds/regimes.')
else:
    print('--- Leave-One-Run-Out Cross-Validation ---')
    cv_result = leave_one_run_out_cv(RUNS, window_size=3, forecast_horizon=2, verbose=True)
    print('\n--- Baseline Comparisons ---')
    baseline_results = evaluate_baselines(RUNS, window_size=3, forecast_horizon=2, verbose=True)
    print_comparison_table(cv_result, baseline_results)


## 6. Signal lead-time analysis

How many epochs *before* detected collapse does each signal deviate from its stable-run
baseline? Positive lead = genuine early warning. Reads the CSVs written in section 4.

In [ ]:
try:
    lag_results = compute_signal_lags(session_dir=SESSION_DIR)
    print_lag_table(lag_results)
except (FileNotFoundError, ValueError) as exc:
    lag_results = {}
    print(f'[signal_lag] {exc}')


## 7. Plots

In [ ]:
import matplotlib.pyplot as plt

# 7.1 Validation-accuracy curves with instability markers.
plot_runs = [{'run_id': r.run_id, 'val_accuracies': r.val_accuracies,
              'instability_epoch': r.instability_epoch,
              'regime': r.run_id.split('__')[0]} for r in RUNS]
plot_val_accuracy_curves(plot_runs); plt.show()


In [ ]:
# 7.2 LORO-CV per-fold metrics.
if 'cv_result' in globals() and cv_result.get('fold_results'):
    plot_cv_metrics(cv_result); plt.show()


In [ ]:
# 7.3 Signal lead-time bar chart.
if lag_results:
    plot_lead_time_bar(lag_results); plt.show()


In [ ]:
# 7.4 RandomForest feature importance (train one predictor on all runs).
agg_seqs = [[canonical_aggregate_features(ep) for ep in r.metrics_sequence] for r in RUNS]
all_keys = sorted({k for seq in agg_seqs for ep in seq for k in ep})
X_all, y_all = [], []
for r, seq in zip(RUNS, agg_seqs):
    Xi, yi = create_sliding_windows(seq, window_size=3, forecast_horizon=2,
                                    instability_epoch=r.instability_epoch,
                                    feature_keys=all_keys, label_mode='detect')
    X_all.extend(Xi); y_all.extend(yi)
if X_all:
    full_predictor = Predictor(window_size=3, forecast_horizon=2, feature_keys=all_keys)
    full_predictor.train(X_all, y_all)
    plot_feature_importance(full_predictor, top_n=20); plt.show()
else:
    print('Not enough windows to train a feature-importance model.')


## 8. Genuine forecasting study (endogenous collapse)

Sections 4-7 are *detection*: those regimes (`delayed_collapse`, `high_learning_rate`)
collapse from exogenous shocks with no internal precursor, so they can only be caught as
they begin. **This section does genuine forecasting** — predicting collapse *before* it
happens (`label_mode='forecast'`, positive windows strictly pre-onset).

That requires *endogenous* collapse: gradual memorization where the internal signals drift
before validation accuracy crashes. We engineer three (augmentation off, tiny/noisy data):
`memorization_collapse` (SimpleCNN, tiny clean data), `label_noise_collapse` (fits clean
labels then memorizes noise), `deep_memorization` (DeepCNN overfits harder), plus `normal`
as a healthy reference.

The research question: **how many epochs ahead can internal signals forecast collapse, and
do they beat simply watching validation accuracy?** Self-contained — runnable on its own.

In [ ]:
import matplotlib.pyplot as plt

# Endogenous collapses degrade from a peak (not to chance) -> chance_level=None and a
# drop detector tuned for gradual degradation. 5 seeds for LORO-CV statistics.
FC_SESSION_DIR, FC_RUNS = run_batch_experiments(
    regimes=('normal', 'memorization_collapse', 'label_noise_collapse', 'deep_memorization'),
    seeds=(100, 101, 102, 103, 104),
    chance_level=None,
    drop_threshold=0.06, drop_window=8, burn_in=8, sustain_epochs=2,
)


In [ ]:
# Val-accuracy curves: confirm the engineered runs peak then degrade (a real onset).
fc_plot_runs = [{'run_id': r.run_id, 'val_accuracies': r.val_accuracies,
                 'instability_epoch': r.instability_epoch,
                 'regime': r.run_id.split('__')[0]} for r in FC_RUNS]
plot_val_accuracy_curves(fc_plot_runs, title='Endogenous collapse: val accuracy'); plt.show()


In [ ]:
# Forecasting skill vs lead time (LORO-CV, forecast mode) vs the val-accuracy baseline.
print('--- Forecasting skill vs lead time ---')
fc_sweep = forecast_horizon_sweep(FC_RUNS, horizons=(1, 2, 3, 5, 8),
                                  window_size=3, verbose=True)
plot_horizon_sweep(fc_sweep, metric='roc_auc'); plt.show()
plot_horizon_sweep(fc_sweep, metric='f1'); plt.show()


In [ ]:
# Which signals move first, and how many epochs before the crash.
print('--- Signal lead-time (which signals precede collapse) ---')
try:
    fc_lag = compute_signal_lags(session_dir=FC_SESSION_DIR)
    print_lag_table(fc_lag)
    if fc_lag:
        plot_lead_time_bar(fc_lag); plt.show()
except (FileNotFoundError, ValueError) as exc:
    print(f'[signal_lag] {exc}')


In [ ]:
# Forecast-mode feature importance: which signals/timesteps drive the forecast.
fc_agg = [[canonical_aggregate_features(ep) for ep in r.metrics_sequence] for r in FC_RUNS]
fc_keys = sorted({k for seq in fc_agg for ep in seq for k in ep})
fc_X, fc_y = [], []
for r, seq in zip(FC_RUNS, fc_agg):
    Xi, yi = create_sliding_windows(seq, window_size=3, forecast_horizon=3,
                                    instability_epoch=r.instability_epoch,
                                    feature_keys=fc_keys, label_mode='forecast')
    fc_X.extend(Xi); fc_y.extend(yi)
if fc_X and sum(fc_y) > 0:
    fc_predictor = Predictor(window_size=3, forecast_horizon=3, feature_keys=fc_keys)
    fc_predictor.train(fc_X, fc_y)
    plot_feature_importance(fc_predictor, top_n=18); plt.show()
else:
    print('No forecastable positive windows at horizon 3 (see the horizon sweep above).')


---
Done. Section 4-7 = detection of exogenous shocks; **section 8 = genuine forecasting of
endogenous collapse**. To scale up, add seeds in section 8 or `regimes=tuple(ALL_REGIMES)`
in section 4.